# SYNAPSE — GoEmotions (separado)

Sección GoEmotions extraída del notebook principal para correr UNA sola vez (independiente de los modelos de malware). Reusa los módulos compartidos.

In [ ]:
MODEL = "GoEmotions"


In [ ]:
# === Profiling: wall-clock time + peak memory (added for R7.8 complexity analysis) ===
import time, sys, resource, csv, os
try:
 import torch as _torch
 _CUDA = _torch.cuda.is_available()
except Exception:
 _CUDA = False

class _Profiler:
 def __init__(self):
 self.rows = []
 self._stack = []
 def _rss_mb(self):
 # ru_maxrss: kilobytes on Linux, bytes on macOS
 m = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
 return m / 1024.0 if sys.platform.startswith("linux") else m / (1024.0 * 1024.0)
 def start(self, name):
 if _CUDA:
 _torch.cuda.synchronize(); _torch.cuda.reset_peak_memory_stats()
 self._stack.append((name, time.perf_counter()))
 def stop(self):
 name, t0 = self._stack.pop()
 if _CUDA:
 _torch.cuda.synchronize()
 dt = time.perf_counter() - t0
 gpu = (_torch.cuda.max_memory_allocated() / 1e6) if _CUDA else 0.0
 row = {"stage": name, "time_s": round(dt, 3),
 "peak_gpu_mb": round(gpu, 1), "proc_maxrss_mb": round(self._rss_mb(), 1)}
 self.rows.append(row)
 print(f"[profile] {name}: {dt:.2f}s | GPU peak {gpu:.0f} MB | proc RSS {row['proc_maxrss_mb']:.0f} MB")
 return row
 def save(self, path):
 d = os.path.dirname(path)
 if d:
 os.makedirs(d, exist_ok=True)
 with open(path, "w", newline="") as f:
 w = csv.DictWriter(f, fieldnames=["stage", "time_s", "peak_gpu_mb", "proc_maxrss_mb"])
 w.writeheader(); w.writerows(self.rows)
 print(f"[profile] saved {len(self.rows)} rows -> {path}")

PROFILE = _Profiler()
PROFILE_T0 = time.perf_counter()
print("[profile] profiler ready; CUDA available:", _CUDA)


In [ ]:
import os
import torch
import json
import logging
import time
import numpy as np
import pandas as pd
import pickle
from transformers import AutoTokenizer
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoModelForSequenceClassification
from torch.utils.data import DataLoader, Dataset
import neurox.data.extraction.transformers_extractor as transformers_extractor
from neurox.data.writer import ActivationsWriter
import neurox.data.loader as data_loader
from transformers import AutoConfig
from tqdm import tqdm
import neurox.interpretation.linear_probe as linear_probe
import neurox.interpretation.utils as utils
import neurox.analysis.visualization as TransformersVisualizer
from sklearn.model_selection import train_test_split
from IPython.display import display
import neurox.interpretation.probeless as probeless
from neurox.interpretation.probeless import (
 get_neuron_ordering,
 get_neuron_ordering_for_all_tags
)
import ast
from torch.cuda.amp import autocast
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import LabelEncoder
from matplotlib_venn import venn2
from neurox.interpretation.linear_probe import get_top_neurons
from sklearn.utils import shuffle

In [ ]:
import logging

# ==========================
# Configure Logging 
# ==========================

logger = logging.getLogger("synapse_logger")
logger.setLevel(logging.INFO)

# Avoid duplicates
if not logger.hasHandlers():

 # Handler 
 file_handler = logging.FileHandler("logs/synapse_extraction_csv_pth.log", mode="w")
 file_handler.setLevel(logging.INFO)

 # Handler 
 console_handler = logging.StreamHandler()
 console_handler.setLevel(logging.INFO)

 # Format
 formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
 file_handler.setFormatter(formatter)
 console_handler.setFormatter(formatter)

 # Add handlers to main logger
 logger.addHandler(file_handler)
 logger.addHandler(console_handler)

logger.info("Logging configured")


In [ ]:
# ==========================
# Revision experiment controls (COST KNOBS) — tune before each run
# ==========================
# Everything new added for the revision runs for the CURRENT MODEL, so switching MODEL
# above and re-running the notebook reruns the full suite (existing attacks + new
# comparisons) for that model. These knobs bound the extra cost.

# Statistical seeds for the multi-seed protocol (R4.4 / R7.3). KEEP AT 1 for the first
# timing run; raise later once per-model wall-clock is known and the budget allows.
N_SEEDS = 5

# Random-neuron control (R1.3 / R4.2): number of random draws for the null distribution.
# 1 makes the p-value/std meaningless, so a few are needed; keep it modest for timing.
RANDOM_CONTROL_DRAWS = 20
SAMPLE_N = 200 # eval-sample size per seed (main time driver)

# Neuron percentages: SAME set as the existing global-silencing sweep (single source of
# truth for the whole pipeline — do NOT invent new ones). New silencing-based cells reuse these.
SWEEP_PCTS = [0.025, 0.05, 0.075, 0.10, 0.125, 0.15, 0.175, 0.20,
 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.65, 0.75, 0.8, 0.95]

# On/off switches for the NEW experiments (set False to skip and save time):
RUN_DETECTION_METRICS = True # R3.2 — cheap (post-processing of predictions)
RUN_RANDOM_CONTROL = True # R1.3/R4.2 — cost ~ RANDOM_CONTROL_DRAWS * len(SWEEP_PCTS) evals
RUN_MEANPOOL_ABLATION = True # R3.4/R7.6 — re-extracts a mean-pool activation set (extraction cost)
RUN_BITFLIP = True # R7.9 — attribution-guided exponent-MSB bit-flip (deterministic; cheap)
RUN_ATTRIBUTION = True # #8 R1.1/R3.3/R7.5 — probe vs conductance/act-grad (captum; HEAVY -> subset)
ATTRIBUTION_N_SAMPLES = 32 # examples for the attribution comparison (bounds conductance cost)
ATTRIBUTION_STEPS = 20 # captum conductance integration steps

print(f"[config] MODEL={MODEL} | N_SEEDS={N_SEEDS} | RANDOM_CONTROL_DRAWS={RANDOM_CONTROL_DRAWS} "
 f"| n_pcts={len(SWEEP_PCTS)} | detection={RUN_DETECTION_METRICS} "
 f"random_ctrl={RUN_RANDOM_CONTROL} meanpool={RUN_MEANPOOL_ABLATION} bitflip={RUN_BITFLIP} attribution={RUN_ATTRIBUTION}")


In [ ]:
def make_cls_silence_hook(indices):
 """
 Forward hook that zeros out the selected neuron indices in the CLS token.
 It is compatible with:
 - BERT / RoBERTa / BigBird / Longformer (tensor output)
 - DistilBERT (tuple output, usually (hidden_state,) or (hidden_state, attentions))
 """
 idxs = torch.tensor(indices, dtype=torch.long)

 def hook(module, inp, output):
 # 1) Unify output into a `hidden` tensor
 if isinstance(output, tuple):
 if len(output) == 0:
 # Nothing to do
 return output
 hidden = output[0]
 else:
 hidden = output

 # 2) Sanity checks
 if not hasattr(hidden, "dim") or hidden.dim() != 3:
 # Not a (batch, seq_len, hidden) tensor do nothing
 return output

 if idxs.numel() == 0:
 # No neuron to silence in this layer
 return output

 # 3) Clone and modify CLS token
 new_hidden = hidden.clone()
 cls_token = new_hidden[:, 0, :] # (batch, hidden_dim)
 cls_token[:, idxs] = 0.0 # silence selected neurons
 new_hidden[:, 0, :] = cls_token

 # 4) Rebuild structure depending on model
 if isinstance(output, tuple):
 # Keep any extra elements (e.g. attentions) untouched
 return (new_hidden,) + tuple(output[1:])
 else:
 return new_hidden

 return hook

In [ ]:
import torch
import os, json, pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report

def find_final_linear_layer(model):
 """
 Find the final Linear layer used for classification.
 Works for BERT, DistilBERT, BigBird, Longformer, etc.
 It selects the Linear whose out_features == config.num_labels.
 """
 if not hasattr(model, "classifier"):
 raise ValueError("Model has no 'classifier' attribute.")

 clf = model.classifier
 num_labels = model.config.num_labels

 # Case 1: classifier is directly a Linear with num_labels outputs
 if hasattr(clf, "weight") and getattr(clf.weight, "shape", None) is not None:
 if clf.weight.shape[0] == num_labels:
 return clf

 # Case 2: classifier is a head module (e.g. BigBird/Longformer)
 for _, module in clf.named_modules():
 if hasattr(module, "weight") and getattr(module.weight, "shape", None) is not None:
 if module.weight.shape[0] == num_labels:
 return module

 raise ValueError("Could not find a final Linear layer with num_labels outputs inside classifier.")

# GoEmotions

## Utilities

In [ ]:
import logging, torch
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score, classification_report

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

def _report(labels_list, preds, tag):
 acc = accuracy_score(labels_list, preds)
 f1w = f1_score(labels_list, preds, average='weighted', zero_division=0)
 logger.info(f"{tag} Accuracy: {acc:.4f}")
 logger.info(f"{tag} Weighted F1: {f1w:.4f}")
 dist = dict(Counter(preds)); logger.info(f"{tag} Pred distribution: {dist}")
 rep = classification_report(labels_list, preds, zero_division=0)
 logger.info(f"{tag} Classification Report:\n{rep}")
 return {"accuracy": acc, "f1_weighted": f1w, "dist": dist, "report": rep, "preds": preds}

def _run_inference_6(model, sample_df, target_ids_tensor, device):
 model.eval()
 preds = []
 with torch.no_grad():
 for _, row in sample_df.iterrows():
 input_ids = torch.tensor(row["input_ids"]).unsqueeze(0).to(device)
 att_mask = torch.tensor(row["attention_mask"]).unsqueeze(0).to(device)
 logits28 = model(input_ids=input_ids, attention_mask=att_mask).logits.squeeze(0)
 logits6 = logits28[target_ids_tensor]
 preds.append(int(torch.argmax(logits6)))
 return preds

In [ ]:
# ===== Utilities =====
from collections import defaultdict
import torch, pandas as pd, logging
logger = logging.getLogger(__name__)

def _map_global_to_layer_indices(model, global_indices, layers_scope="all"):
 """Map global neuron ids [0..L*H) to {layer_idx: [local_ids]}."""
 H = model.config.hidden_size
 L = model.config.num_hidden_layers
 out = defaultdict(list)
 for g in global_indices:
 li, ji = g // H, g % H
 if layers_scope == "last_only"and li != L - 1:
 continue
 out[li].append(int(ji))
 return out

def get_classifier_linear_goemo(model):
 """Return the final linear layer (28 x H) for GoEmotions."""
 clf = getattr(model, "classifier", None)
 if clf is not None and hasattr(clf, "weight"):
 return clf
 raise RuntimeError("Classifier linear layer not found.")

## Dataset Configuration

In [ ]:
# Dataset and mappings
GOEMOTIONS_PATH = "data/goemotions"
INPUT_FILE = f"{GOEMOTIONS_PATH}/test.tsv"
EMOTIONS_FILE = f"{GOEMOTIONS_PATH}/emotions.txt"

# Target emotions (subset of original GoEmotions)
TARGET_EMOTIONS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]

# Pretrained Model
GOEMOTIONS_MODEL_HF = "monologg/bert-base-cased-goemotions-original"

# Outputs
SAMPLE_OUTPUT = f"{GOEMOTIONS_PATH}/sample_60.json"
TOKENIZED_OUTPUT = f"{GOEMOTIONS_PATH}/tokenized.pt"
LABELS_OUTPUT = f"{GOEMOTIONS_PATH}/labels.pt"
LABEL_MAPPING_OUTPUT = f"{GOEMOTIONS_PATH}/label_mapping.json"
CSV_REPORT_PATH = f"{GOEMOTIONS_PATH}/classification_report_eval.csv"
CSV_REPORT_GOBAL_SILENCING = f"{GOEMOTIONS_PATH}/classification_report_global_silencing.csv"
ACTIVATIONS_GOEMOTIONS = f"{GOEMOTIONS_PATH}/activations.json"
SAMPLE_OUTPUT_JSON = "data/goemotions/sample_df.json"
# Device

device_goemo = torch.device("cuda"if torch.cuda.is_available() else "cpu")

In [ ]:
def silence_top_global_percentage_and_evaluate(
 model,
 sample_df,
 labels_list,
 probe,
 label2idx,
 percentage=None,
 experiment_title=None,
 report_path=None,
 custom_indices=None # Nuevo: permite pasar neuronas custom (por ejemplo aleatorias)
):
 import os
 import torch
 import pandas as pd
 from sklearn.metrics import accuracy_score, f1_score, classification_report

 hidden_dim = model.config.hidden_size
 num_layers = model.config.num_hidden_layers
 total_neurons = num_layers * hidden_dim

 # Decide qué neuronas silenciar
 if custom_indices is not None:
 top_neurons = custom_indices
 print(f"Silencing custom list of {len(top_neurons)} neurons")
 else:
 # Si no, selecciona top del probe (como siempre)
 top_neurons = top_neurons_probe(
 probe, percentage=percentage, class_to_idx=label2idx
 )
 print(f"Silencing {len(top_neurons)} neurons from probe (percentage={percentage})")

 # Hook setup
 encoder_layers = get_encoder_layers(model)
 hook_handles = []
 for i in range(num_layers):
 # Neuronas de esta capa
 indices_layer = [idx - i * hidden_dim for idx in top_neurons if i * hidden_dim <= idx < (i + 1) * hidden_dim]
 if indices_layer:
 print(f"Layer {i}: silencing {len(indices_layer)} neurons")
 if hasattr(encoder_layers[i], "output"):
 handle = encoder_layers[i].output.register_forward_hook(make_cls_silence_hook(indices_layer))
 else:
 handle = encoder_layers[i].register_forward_hook(make_cls_silence_hook(indices_layer))
 hook_handles.append(handle)

 # Evaluación estándar (igual que ya tienes)
 model.eval()
 predictions = []
 for i in range(len(sample_df)):
 input_ids_tensor = torch.tensor(sample_df.loc[i, 'input_ids']).unsqueeze(0).to(model.device)
 attention_mask_tensor = torch.tensor(sample_df.loc[i, 'attention_mask']).unsqueeze(0).to(model.device)
 with torch.no_grad():
 outputs = model(input_ids=input_ids_tensor, attention_mask=attention_mask_tensor)
 logits = outputs['logits']
 pred = torch.argmax(logits, dim=1).item()
 predictions.append(pred)
 del input_ids_tensor, attention_mask_tensor, outputs, logits
 torch.cuda.empty_cache()

 # Métricas y reporte
 accuracy = accuracy_score(labels_list, predictions)
 f1 = f1_score(labels_list, predictions, average='weighted')
 report_dict = classification_report(labels_list, predictions, output_dict=True)
 report_df = pd.DataFrame(report_dict).transpose().round(4)
 report_df = report_df.drop("accuracy", errors="ignore")

 accuracy_row = pd.DataFrame({
 'precision': [""],
 'recall': [""],
 'f1-score': [accuracy],
 'support': [sum(report_df["support"])]
 }, index=["overall_accuracy"])
 final_df = pd.concat([report_df, accuracy_row])

 # Guardar reporte
 if report_path is None:
 report_path = "results/class_silencing_global.csv"
 os.makedirs(os.path.dirname(report_path), exist_ok=True)

 if experiment_title is None:
 experiment_title = "Silencing top global neurons"
 if not os.path.exists(report_path):
 with open(report_path, "w") as f:
 f.write(f"# {experiment_title}\n")
 final_df.to_csv(f)
 else:
 with open(report_path, "a") as f:
 f.write(f"\n\n# {experiment_title}\n")
 final_df.to_csv(report_path, mode="a")

 print(f"Accuracy after silencing: {accuracy:.4f}")
 print(f"Weighted F1 Score: {f1:.4f}")
 print(f"Classification report saved to {report_path}")

 # Quitar hooks
 for handle in hook_handles:
 handle.remove()
 print("All hooks removed after evaluation")

In [ ]:
if not os.path.exists(SAMPLE_OUTPUT):
 # Load emotion names
 with open(EMOTIONS_FILE, "r") as f:
 id2emotion = [line.strip() for line in f.readlines()]
 emotion2id = {e: i for i, e in enumerate(id2emotion)}

 # Select target emotions and their GoEmotions IDs
 TARGET_EMOTIONS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]
 target_ids = [emotion2id[e] for e in TARGET_EMOTIONS]

 # Mapping from GoEmotion ID to 0–5 label
 goemo2local = {eid: i for i, eid in enumerate(target_ids)}

 # Load dataset
 df = pd.read_csv(INPUT_FILE, sep="\t", header=None, names=["text", "labels", "split"])
 df = df.dropna(subset=["labels"])
 df["label_ids"] = df["labels"].apply(lambda x: list(map(int, str(x).split(","))))

 # Filter: single-label only & target emotions
 df_filtered = df[df["label_ids"].apply(lambda ids: len(ids) == 1 and ids[0] in target_ids)].copy()
 df_filtered["label_id"] = df_filtered["label_ids"].apply(lambda ids: goemo2local[ids[0]])

 # Count examples per class
 counts = df_filtered["label_id"].value_counts()
 print("Available examples for selected emotions:")
 print(counts)

 # Balanced subset (max 10 per class)
 max_per_class = 10
 samples = []

 for label in counts.index:
 subset = df_filtered[df_filtered["label_id"] == label]
 sampled = shuffle(subset, random_state=42).iloc[:max_per_class]
 samples.append(sampled[["text", "label_id"]])

 df_final = pd.concat(samples).reset_index(drop=True)

 # Save to JSON
 df_final.to_json(SAMPLE_OUTPUT, orient="records", lines=True, force_ascii=False)
 print(f"\n Saved dataset: {len(df_final)} examples (max {max_per_class} per emotion)")
else:
 print(f"Skipping dataset generation: {SAMPLE_OUTPUT} already exists.")

## Original Performance

### Load model, tokenizer and inputs

In [ ]:
# Load dataset
with open(SAMPLE_OUTPUT, "r") as f:
 data = [json.loads(line) for line in f]

texts = [x["text"] for x in data]
labels = [x["label_id"] for x in data]

# Label mappings
label2id = {label: i for i, label in enumerate(sorted(set(labels)))}
id2label = {i: label for label, i in label2id.items()}
label_ids = [label2id[label] for label in labels]

# Tokenize and save only if not already saved
tokenizer = AutoTokenizer.from_pretrained(GOEMOTIONS_MODEL_HF)

if not os.path.exists(TOKENIZED_OUTPUT):
 encodings = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
 torch.save(encodings, TOKENIZED_OUTPUT)
 logger.info("Tokenized inputs saved.")
else:
 logger.warning(f"Skipping: {TOKENIZED_OUTPUT} already exists.")

if not os.path.exists(LABELS_OUTPUT):
 torch.save(torch.tensor(label_ids), LABELS_OUTPUT)
 logger.info("Label tensor saved.")
else:
 logger.warning(f"Skipping: {LABELS_OUTPUT} already exists.")

if not os.path.exists(LABEL_MAPPING_OUTPUT):
 with open(LABEL_MAPPING_OUTPUT, "w") as f:
 json.dump(label2id, f, indent=2)
 logger.info("Label mapping saved.")
else:
 logger.warning(f"Skipping: {LABEL_MAPPING_OUTPUT} already exists.")

# NEW: Generate sample_df for later neuron silencing evaluation
if not os.path.exists(SAMPLE_OUTPUT_JSON):
 logger.info("Creating and saving sample_df.json for evaluation hooks...")
 sample_rows = []
 for text in texts:
 encoded = tokenizer(text, truncation=True, padding="max_length", max_length=128)
 sample_rows.append({
 "input_ids": encoded["input_ids"],
 "attention_mask": encoded["attention_mask"]
 })
 sample_df = pd.DataFrame(sample_rows)
 sample_df.to_json(SAMPLE_OUTPUT_JSON, orient="records", lines=True)
 logger.info(f"sample_df saved to {SAMPLE_OUTPUT_JSON}")
else:
 logger.warning(f"Skipping: {SAMPLE_OUTPUT_JSON} already exists.")

# Summary
logger.info("Emotions (IDs): %s", sorted(label2id.keys()))
logger.info("Label mapping: %s", label2id)

## Inference

In [ ]:
from tqdm import tqdm
import torch

# Define the target GoEmotions IDs
target_emotion_names = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]

# Load emotion mapping
with open(EMOTIONS_FILE, "r") as f:
 id2emotion = [line.strip() for line in f.readlines()]
emotion2id = {e: i for i, e in enumerate(id2emotion)}

target_ids = [emotion2id[e] for e in target_emotion_names]
target_ids_tensor = torch.tensor(target_ids).to(device_goemo)

# Map GoEmotions IDs local labels
label2id = {goid: i for i, goid in enumerate(target_ids)}
id2label = {i: goid for goid, i in label2id.items()}

print(f"Target GoEmotions IDs: {target_ids}")
print(f"Mapping to local labels: {label2id}")

# Load model_goem
from transformers import AutoModelForSequenceClassification
model_goem = AutoModelForSequenceClassification.from_pretrained(GOEMOTIONS_MODEL_HF)
model_goem.to(device_goemo)
model_goem.eval()

# Load data
inputs = torch.load(TOKENIZED_OUTPUT, weights_only=False)
labels = torch.load(LABELS_OUTPUT).tolist()

predictions = []
true_labels = []

with torch.no_grad():
 for i in tqdm(range(len(labels))):
 input_ids = inputs["input_ids"][i].unsqueeze(0).to(device_goemo)
 attention_mask = inputs["attention_mask"][i].unsqueeze(0).to(device_goemo)

 logits = model_goem(input_ids=input_ids, attention_mask=attention_mask).logits.squeeze(0)

 selected_logits = logits[target_ids_tensor]
 pred_local = torch.argmax(selected_logits).item()

 predictions.append(pred_local)
 true_labels.append(labels[i]) # already 0–5



In [ ]:
# Report
accuracy = accuracy_score(true_labels, predictions)
f1 = f1_score(true_labels, predictions, average="weighted")
ordered_labels = sorted(label2id.values())

report = classification_report(
 true_labels,
 predictions,
 labels=ordered_labels,
 target_names=[id2label[i] for i in ordered_labels],
 output_dict=True,
 zero_division=0
)

report_df = pd.DataFrame(report).transpose().round(4)
accuracy_row = pd.DataFrame({
 'precision': [""],
 'recall': [""],
 'f1-score': [accuracy],
 'support': [sum(report_df["support"])]
}, index=["overall_accuracy"])

final_df = pd.concat([report_df, accuracy_row])

if not os.path.exists(CSV_REPORT_PATH):
 final_df.to_csv(CSV_REPORT_PATH)
 print(f"Report saved to {CSV_REPORT_PATH}")
else:
 print(f"Skipping save: {CSV_REPORT_PATH} already exists.")

print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")

## Dataset Wrapper and DataLoader (Goemotions)

In [ ]:
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

class GoEmotionsDataset(Dataset):
 def __init__(self, input_ids, labels):
 self.input_ids = input_ids
 self.labels = labels

 def __len__(self):
 return len(self.input_ids)

 def __getitem__(self, idx):
 return torch.tensor(self.input_ids[idx]), torch.tensor(self.labels[idx])

# Load input IDs and labels from disk
input_data = torch.load(TOKENIZED_OUTPUT, weights_only=False)
labels = torch.load(LABELS_OUTPUT).tolist()

input_ids_list = input_data["input_ids"].tolist()

# Create dataset and dataloader
dataset = GoEmotionsDataset(input_ids_list, labels)
dataloader = DataLoader(dataset, batch_size=4, shuffle=False)

logger.info("Dataloader created successfully.")

## Extract Activations

In [ ]:
PROFILE.start("GoEmotions: activation extraction")
if os.path.exists(ACTIVATIONS_GOEMOTIONS):
 logger.info(f"Activations already exist at {ACTIVATIONS_GOEMOTIONS}. Skipping extraction.")
else:
 logger.info("Starting activation extraction from model (CLS token only).")
 
 transformers_extractor.extract_representations(
 model=model_goem,
 input_tokens_list=input_ids_list, 
 output_file=ACTIVATIONS_GOEMOTIONS,
 device=device_goemo,
 output_type="json", 
 decompose_layers=False,
 filter_layers=None
 )

 logger.info(f"Activations successfully saved to {ACTIVATIONS_GOEMOTIONS}")
PROFILE.stop()


## Load Activations

In [ ]:
import torch
import numpy as np
import logging
from collections import defaultdict
from sklearn.preprocessing import LabelEncoder

logger = logging.getLogger(__name__)

def create_tensors_goemo(tokens_data, activations, task_specific_tag="NN", task_type="classification", dtype=torch.float32):
 """
 Create input/output tensors from CLS activations and labels for classification tasks.

 Args:
 tokens_data (list): List of dicts with keys "tokens"and "target"
 activations (list): List of numpy arrays with CLS activations
 task_specific_tag (str): Not used for CLS, kept for compatibility
 task_type (str): "classification"or "regression"
 dtype (torch.dtype): Data type of the tensors

 Returns:
 X (torch.Tensor): Input features (num_samples, num_layers * hidden_size)
 y (torch.Tensor): Labels
 mapping (tuple): label2idx, idx2label, None, None
 """

 logger.info("Creating tensors from activations and labels")

 # Number of samples
 num_samples = len(tokens_data)
 assert num_samples == len(activations), "Mismatch between tokens and activations"

 logger.info(f"Number of samples: {num_samples}")

 # Flatten each activation: (num_layers, 1, hidden_dim) (num_layers * hidden_dim)
 X = []
 for i, sample in enumerate(activations):
 if sample.ndim == 3 and sample.shape[1] == 1:
 flattened = sample.squeeze(1).flatten()
 elif sample.ndim == 2:
 flattened = sample.flatten()
 else:
 raise ValueError(f"Unexpected shape for activation {i}: {sample.shape}")
 X.append(flattened)
 X = np.array(X)

 
 # Encode labels
 labels = [sample["target"] for sample in tokens_data]
 label_encoder = LabelEncoder()
 y = label_encoder.fit_transform(labels)

 # Logging label mapping
 label2idx = {label: int(idx) for idx, label in enumerate(label_encoder.classes_)}
 idx2label = {int(idx): label for label, idx in label2idx.items()}
 logger.info(f"Labels mapping: {label2idx}")

 return (
 torch.tensor(X, dtype=dtype),
 torch.tensor(y),
 (label2idx, idx2label, None, None)
 )

In [ ]:
from neurox.data.loader import load_activations
from neurox.interpretation import utils

# Load activations
activations, num_layers = load_activations(ACTIVATIONS_GOEMOTIONS)
logger.info(f"Activations loaded from {ACTIVATIONS_GOEMOTIONS} with {num_layers} layers")

# Prepare dataset with correct structure
sentence_data = [{"tokens": ["[CLS]"], "target": label} for label in labels]

# Convert to tensors
X, y, mapping = create_tensors_goemo(
 sentence_data,
 activations,
 task_specific_tag="NN",
 task_type="classification"
)

label2idx, idx2label, _, _ = mapping
logger.info("Tensors and label mappings created successfully")

## Train Probe

In [ ]:
PROFILE.start("GoEmotions: probe training + ranking")
# Convert tensors to numpy arrays (required by train_logistic_regression_probe)
X_np = X.numpy() if isinstance(X, torch.Tensor) else X
y_np = y.numpy() if isinstance(y, torch.Tensor) else y

# Train logistic regression probe
logger.info("Training logistic regression probe")
probe = linear_probe.train_logistic_regression_probe(
 X_np, y_np,
 lambda_l1=1.1,
 lambda_l2=1.1
)

# Evaluate the trained probe
logger.info("Evaluating the probe")
scores = linear_probe.evaluate_probe(probe, X_np, y_np, idx_to_class=idx2label)
logger.info(f"Probe evaluation results:\n{scores}")

# Get top neurons
top_neurons_probe, per_class_top_neurons = linear_probe.get_top_neurons(
 probe,
 percentage=0.1,
 class_to_idx=label2idx
)
logger.info(f"Top global neurons: {top_neurons_probe}")
logger.info(f"Top neurons per class: {per_class_top_neurons}")
PROFILE.stop()


## Silencing Functions

In [ ]:
# Load tokenized input and labels
sample_df = pd.read_json(SAMPLE_OUTPUT_JSON, lines=True)
labels_list = torch.load(LABELS_OUTPUT).tolist()

In [ ]:
import torch

def make_cls_silence_hook(indices: list[int]):
 # crea un tensor de índices (vacío si no hay nada que silenciar)
 indices_tensor = torch.tensor(indices, dtype=torch.long) if indices else torch.tensor([], dtype=torch.long)

 def hook(module, input, output):
 # sólo intervenimos si es un Tensor
 if not isinstance(output, torch.Tensor):
 return output

 out = output.clone()
 idx = indices_tensor.to(out.device)

 if out.dim() == 3:
 # batch × seq_len × hidden
 cls = out[:, 0, :] # (batch, hidden)
 mask = torch.ones_like(cls)
 if idx.numel() > 0:
 mask[:, idx] = 0.0
 out[:, 0, :] = cls * mask
 elif out.dim() == 2:
 # batch × hidden (ej. pooler)
 mask = torch.ones_like(out)
 if idx.numel() > 0:
 mask[:, idx] = 0.0
 out = out * mask

 return out

 return hook

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# 2⃣ Celda: Entrenar el probe y extraer los TOP_NEURONS
# ──────────────────────────────────────────────────────────────────────────────

# (a) Entrena tu logistic regression probe como antes
probe = linear_probe.train_logistic_regression_probe(
 X_np, y_np,
 lambda_l1=0.001,
 lambda_l2=0.001
)

# (b) Obtén los índices globales de las neuronas más relevantes
# `top_neurons_probe` es una lista de enteros en [0, num_layers*hidden_size)
top_neurons_probe, per_class_top_neurons = linear_probe.get_top_neurons(
 probe,
 percentage=0.10, # 10% de las neuronas
 class_to_idx=label2idx
)

# (c) Ahora sí definimos `global_indices` para usar en el hook
global_indices = top_neurons_probe

# Comprueba un par de valores:
print(f"Número total de neuronas a silenciar: {len(global_indices)}")
print(f"Primeros 10 índices globales: {global_indices[:10]}") # deben estar entre 0 y hidden_size*num_layers-1

## [CLS] vs mean-pool ranking agreement — R3.4 / R7.6 / R4.3 (full, GoEmotions)


In [ ]:
# === [CLS] vs mean-pool ranking agreement (FULL) on GoEmotions — R3.4 / R7.6 / R4.3 ===
# GoEmotions has short sequences, so the full ablation lives here: re-extract a
# mean-pooled representation, train its probe, and compare the neuron-importance ranking
# against the [CLS] probe (Spearman / Kendall + top-k overlap). NO re-silencing (option c).
# Gated by RUN_MEANPOOL_ABLATION.
import ranking_agreement, pandas as pd

if not RUN_MEANPOOL_ABLATION:
 print("[R3.4/R7.6] GoEmotions mean-pool agreement skipped (RUN_MEANPOOL_ABLATION=False)")
else:
 _act_mean = ACTIVATIONS_GOEMOTIONS.replace(".json", "_mean.json")
 transformers_extractor.extract_representations(
 model=model_goem,
 input_tokens_list=input_ids_list,
 output_file=_act_mean,
 device=device_goemo,
 output_type="json",
 decompose_layers=False,
 filter_layers=None,
 pooling="mean",
 )
 _acts_mean, _ = load_activations(_act_mean)
 _sent_mean = [{"tokens": ["[MEAN]"], "target": label} for label in labels]
 X_mean, y_mean, _ = create_tensors_goemo(
 _sent_mean, _acts_mean, task_specific_tag="NN", task_type="classification"
 )
 X_mean_np = X_mean.numpy() if isinstance(X_mean, torch.Tensor) else X_mean
 y_mean_np = y_mean.numpy() if isinstance(y_mean, torch.Tensor) else y_mean
 probe_mean = linear_probe.train_logistic_regression_probe(
 X_mean_np, y_mean_np, lambda_l1=0.001, lambda_l2=0.001
 )

 imp_cls = ranking_agreement.probe_importance(probe) # GoEmotions [CLS] ranking (canonical probe, lambda 0.001)
 imp_mean = ranking_agreement.probe_importance(probe_mean) # GoEmotions mean-pool ranking
 agr = ranking_agreement.compare_rankings(imp_cls, imp_mean, top_k_frac=0.10)
 print("[R3.4/R7.6] GoEmotions [CLS] vs mean-pool ranking agreement:", agr)

 _goemo_dir = os.path.dirname(ACTIVATIONS_GOEMOTIONS) # e.g. data/goemotions
 os.makedirs(f"{_goemo_dir}/results", exist_ok=True)
 _agr_out = f"{_goemo_dir}/results/cls_vs_meanpool_agreement_goemotions.csv"
 pd.DataFrame([agr]).to_csv(_agr_out, index=False)
 print("[R3.4/R7.6] saved ->", _agr_out)


In [ ]:
from collections import defaultdict

hidden_size = model_goem.config.hidden_size # p.ej. 768
num_layers = model_goem.config.num_hidden_layers # p.ej. 12

layer_to_indices = defaultdict(list)
for gidx in global_indices:
 layer_idx = gidx // hidden_size
 neuron_idx = gidx % hidden_size
 layer_to_indices[layer_idx].append(int(neuron_idx))

# Verifica que todo esté correcto
for L in sorted(layer_to_indices):
 print(f"Capa {L}: {len(layer_to_indices[L])} neuronas")

In [ ]:
handles = [] # para luego removerlos

for layer_idx, layer in enumerate(model_goem.bert.encoder.layer):
 idxs = layer_to_indices.get(layer_idx, [])
 if idxs:
 h = layer.output.LayerNorm.register_forward_hook(
 make_cls_silence_hook(idxs)
 )
 handles.append(h)

In [ ]:
# Si quieres además silenciar en la salida del pooler:
final_layer_idxs = layer_to_indices.get(num_layers-1, [])
if final_layer_idxs:
 h = model_goem.bert.pooler.dense.register_forward_hook(
 make_cls_silence_hook(final_layer_idxs)
 )
 handles.append(h)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
import torch

def evaluate_silenced_model(
 model,
 sample_df,
 labels_list,
 target_ids_tensor,
 device,
 label2idx
):
 """
 Corre inferencia en el modelo (con hooks ya activos),
 selecciona sólo los logits de los target_ids_tensor,
 calcula accuracy, f1 y classification_report.
 """
 model.eval()
 preds, trues = [], []
 with torch.no_grad():
 for i, row in sample_df.iterrows():
 input_ids = torch.tensor(row["input_ids"]).unsqueeze(0).to(device)
 attention_mask = torch.tensor(row["attention_mask"]).unsqueeze(0).to(device)

 logits = model(input_ids=input_ids, attention_mask=attention_mask).logits.squeeze(0)
 sel_logits = logits[target_ids_tensor]
 pred_local = torch.argmax(sel_logits).item()

 preds.append(pred_local)
 trues.append(labels_list[i])

 acc = accuracy_score(trues, preds)
 f1 = f1_score(trues, preds, average="weighted", zero_division=0)
 report_dict = classification_report(
 trues,
 preds,
 labels=list(label2idx.values()),
 target_names=[str(k) for k in sorted(label2idx.keys())],
 zero_division=0,
 output_dict=True
 )
 return {"accuracy": acc, "f1": f1, "report": report_dict}

## GoEmotions: random-neuron control (R1.3/R4.2) + bit-flip (R7.9)


In [ ]:
# === GoEmotions: random-neuron control (R1.3/R4.2) + bit-flip (R7.9) ===
# Mirrors the malware versions on the GoEmotions model (model_goem), reusing its own
# silencing machinery (evaluate_silenced_model, make_cls_silence_hook, hooks on
# layer.output.LayerNorm + pooler) and the shared modules. Detection metrics (R3.2) are
# NOT applied here (GoEmotions is not a detection task). Gated by RUN_RANDOM_CONTROL / RUN_BITFLIP.
import perturbation_stats, bitflip_attack, pandas as pd
from collections import defaultdict

_goemo_dir = os.path.dirname(ACTIVATIONS_GOEMOTIONS)
os.makedirs(f"{_goemo_dir}/results", exist_ok=True)
_hidden_g = model_goem.config.hidden_size
_nlayers_g = model_goem.config.num_hidden_layers
_total_g = _hidden_g * _nlayers_g

def _goemo_eval_f1(custom_indices):
 """Weighted-F1 for model_goem with `custom_indices` (global neuron ids) silenced on CLS."""
 l2i = defaultdict(list)
 for g in custom_indices:
 l2i[int(g) // _hidden_g].append(int(g) % _hidden_g)
 handles = []
 for li, idxs in l2i.items():
 if idxs:
 handles.append(
 model_goem.bert.encoder.layer[li].output.LayerNorm.register_forward_hook(
 make_cls_silence_hook(idxs)))
 final = l2i.get(_nlayers_g - 1, [])
 if final:
 handles.append(model_goem.bert.pooler.dense.register_forward_hook(make_cls_silence_hook(final)))
 try:
 sc = evaluate_silenced_model(model_goem, sample_df, labels_list,
 target_ids_tensor, device_goemo, label2idx)
 finally:
 for h in handles:
 h.remove()
 return sc["f1"]

def _goemo_top(pct):
 top, _ = linear_probe.get_top_neurons(probe, percentage=pct, class_to_idx=label2idx)
 return list(top)

# --- Random-neuron control (R1.3 / R4.2) ---
if not RUN_RANDOM_CONTROL:
 print("[R1.3][GoEmo] skipped (RUN_RANDOM_CONTROL=False)")
else:
 _base = _goemo_eval_f1([])
 print(f"[R1.3][GoEmo] baseline weighted-F1 = {_base:.4f} (total neurons = {_total_g})")
 _rows = []
 for pct in SWEEP_PCTS:
 r = perturbation_stats.top_vs_random(_goemo_eval_f1, _base, _goemo_top(pct), _total_g,
 n_seeds=RANDOM_CONTROL_DRAWS, base_seed=0, higher_is_better=True)
 r["percentage"] = pct; _rows.append(r)
 print(f"[R1.3][GoEmo] {pct:.0%}: top_drop={r['top_drop']:.4f} "
 f"random={r['random_mean_drop']:.4f}+/-{r['random_std_drop']:.4f} p={r['p_empirical']:.4f}")
 _cols = ["percentage", "k", "baseline", "top_metric", "top_drop", "random_mean_metric",
 "random_mean_drop", "random_std_drop", "p_empirical", "z_score", "n_seeds"]
 pd.DataFrame([{c: r[c] for c in _cols} for r in _rows]).to_csv(
 f"{_goemo_dir}/results/random_control_goemotions.csv", index=False)
 print("[R1.3][GoEmo] saved ->", f"{_goemo_dir}/results/random_control_goemotions.csv")

# --- Attribution-guided bit-flip (R7.9), same literature basis as malware ---
if not RUN_BITFLIP:
 print("[R7.9][GoEmo] skipped (RUN_BITFLIP=False)")
else:
 _out_proj = find_final_linear_layer(model_goem)
 _orig = _out_proj.weight.data.clone()
 _base = evaluate_silenced_model(model_goem, sample_df, labels_list,
 target_ids_tensor, device_goemo, label2idx)["f1"]
 print(f"[R7.9][GoEmo] baseline weighted-F1 = {_base:.4f}")
 _rows = []
 for pct in SWEEP_PCTS:
 _cf = sorted({int(g) % _hidden_g for g in _goemo_top(pct)})
 try:
 _nb = bitflip_attack.flip_exponent_msb_columns_(_out_proj.weight.data, _cf)
 _f1 = evaluate_silenced_model(model_goem, sample_df, labels_list,
 target_ids_tensor, device_goemo, label2idx)["f1"]
 finally:
 _out_proj.weight.data.copy_(_orig)
 _rows.append({"percentage": pct, "n_cols": len(_cf), "n_bits_flipped": _nb,
 "baseline_f1": _base, "f1_after": _f1, "f1_drop": _base - _f1})
 print(f"[R7.9][GoEmo] {pct:.0%}: cols={len(_cf)} f1 {_base:.4f}->{_f1:.4f}")
 pd.DataFrame(_rows).to_csv(f"{_goemo_dir}/results/bitflip_goemotions.csv", index=False)
 print("[R7.9][GoEmo] saved ->", f"{_goemo_dir}/results/bitflip_goemotions.csv")


## Attacks

### Global silencing

In [ ]:
import torch
from collections import defaultdict

def silence_top_global_percentage_and_evaluate(
 model,
 sample_df,
 labels_list,
 probe,
 label2idx,
 percentage=0.10,
 experiment_title="Silencing neurons",
 report_path=None,
 custom_indices=None # <-- nuevo parámetro
):
 # ① Selección de índices
 if custom_indices is not None:
 top_neurons = custom_indices
 print(f"Silencing custom list of {len(top_neurons)} neurons")
 else:
 # si no se pasa custom_indices, volvemos a llamar al probe (no existe shadowing aquí)
 top_neurons, _ = linear_probe.get_top_neurons(
 probe, percentage=percentage, class_to_idx=label2idx
 )
 print(f"Silencing {len(top_neurons)} neurons from probe (percentage={percentage})")

 # ② Mapeo globalpor capa
 hidden_size = model.config.hidden_size
 layer_to_indices = defaultdict(list)
 for gidx in top_neurons:
 layer_idx = gidx // hidden_size
 neuron_idx = gidx % hidden_size
 layer_to_indices[layer_idx].append(int(neuron_idx))

 # ③ Registro de hooks
 handles = []
 for layer_idx, indices in layer_to_indices.items():
 if not indices:
 continue
 # hook en la LayerNorm de cada encoder.layer
 h = model.bert.encoder.layer[layer_idx].output.LayerNorm.register_forward_hook(
 make_cls_silence_hook(indices)
 )
 handles.append(h)

 # (Opcional) pooler
 final_idxs = layer_to_indices.get(model.config.num_hidden_layers-1, [])
 if final_idxs:
 h = model.bert.pooler.dense.register_forward_hook(
 make_cls_silence_hook(final_idxs)
 )
 handles.append(h)

 # ─── ④ Evaluación ────────────────────────────────────────────────
 from pathlib import Path
 # asegúrate de tener target_ids_tensor disponible en el scope
 silence_scores = evaluate_silenced_model(
 model,
 sample_df,
 labels_list,
 target_ids_tensor, # tu tensor con los IDs GoEmotions
 device_goemo,
 label2idx
 )

 # guardar el classification_report en CSV
 report_df = (
 pd.DataFrame(silence_scores["report"])
 .transpose()
 .round(4)
 )
 # añadir fila de accuracy/f1 global si quieres
 report_df.to_csv(report_path, index=True)
 print(f"Accuracy after silencing: {silence_scores['accuracy']:.4f}")
 print(f"F1 Score after silencing: {silence_scores['f1']:.4f}")

 # ─── ⑤ Remove hooks ──────────────────────────────────────────────
 for h in handles:
 h.remove()
 print("All hooks removed")

In [ ]:
# tras cargar sample_df, labels_list y probe como hacías
silence_top_global_percentage_and_evaluate(
 model=model_goem,
 sample_df=sample_df,
 labels_list=labels_list,
 probe=probe,
 label2idx=label2idx,
 percentage=0.6,
 experiment_title="Silencing 10% Global Neurons",
 report_path=CSV_REPORT_GOBAL_SILENCING
)

In [ ]:
for h in handles:
 h.remove()

### Label-specific silencing (impact per class)

In [ ]:
def get_top_k_neurons_for_class_exact_goemo(probe, percentage: float, class_to_idx: dict, class_id: int) -> list[int]:
 """
 Top-k neurons most important for a specific class (absolute weight).
 Works with either a torch Linear probe (linear.weight) or sklearn-like (coef_).
 """
 # Extract weight matrix [num_classes, num_neurons]
 if hasattr(probe, "linear") and hasattr(probe.linear, "weight"):
 W = probe.linear.weight.detach().abs().cpu().numpy()
 elif hasattr(probe, "coef_"):
 W = np.abs(np.asarray(probe.coef_))
 else:
 raise RuntimeError("Probe weights not found (expected .linear.weight or .coef_)")

 cidx = class_to_idx.get(class_id, class_id)
 class_w = W[cidx] # [num_neurons]
 total_neurons = class_w.shape[0]
 top_n = max(1, round(percentage * total_neurons))
 idx = np.argpartition(class_w, top_n * -1)[-top_n:]
 idx = idx[np.argsort(-class_w[idx])]
 return idx.tolist()

def get_encoder_layers(model):
 if hasattr(model, "bert"):
 return model.bert.encoder.layer
 elif hasattr(model, "longformer"):
 return model.longformer.encoder.layer
 elif hasattr(model, "distilbert"):
 return model.distilbert.transformer.layer
 else:
 raise NotImplementedError("Unsupported model architecture.")

In [ ]:
def silence_top_class_percentage_and_evaluate_goemo(
 model,
 sample_df,
 labels_list,
 probe,
 label2idx, # local 0–5 mapping used to index probe rows
 target_ids_tensor, # tensor with the 6 original GoEmotions IDs (to slice logits)
 class_id: int,
 percentage: float = 0.10,
 report_path: str = None,
 experiment_title: str = None
):
 """
 Per-class neuron silencing for GoEmotions (6-class slice).
 Selects top-k by class from the probe and zeros those dims via forward hooks.
 """
 hidden_dim = model.config.hidden_size
 num_layers = model.config.num_hidden_layers

 # Top-k global neuron indices for the given class
 top_class_neurons = get_top_k_neurons_for_class_exact_goemo(
 probe, percentage=percentage, class_to_idx=label2idx, class_id=class_id
 )
 logger.info(f"Silencing {len(top_class_neurons)} neurons for class {class_id} ({percentage:.2%} of total)")

 # Save selected neuron indices (optional but useful)
 neurons_dir = os.path.join(GOEMOTIONS_PATH, "neurons")
 os.makedirs(neurons_dir, exist_ok=True)
 json_path = os.path.join(neurons_dir, f"top_{int(percentage*100)}p_neurons_class_{class_id}.json")
 with open(json_path, "w") as f:
 json.dump(top_class_neurons, f, indent=2)
 logger.info(f"Saved neuron indices to {json_path}")

 # Register hooks per encoder layer
 encoder_layers = get_encoder_layers(model)
 hook_handles = []
 for i in range(num_layers):
 layer_indices = [idx - i * hidden_dim for idx in top_class_neurons
 if i * hidden_dim <= idx < (i + 1) * hidden_dim]
 if layer_indices:
 logger.info(f"Layer {i}: silencing {len(layer_indices)} neurons for class {class_id}")
 handle = encoder_layers[i].output.register_forward_hook(make_cls_silence_hook(layer_indices))
 hook_handles.append(handle)

 # Inference over the 6-way slice (only selected GoEmotions IDs)
 model.eval()
 predictions = []
 for i in range(len(sample_df)):
 input_ids = torch.tensor(sample_df.loc[i, 'input_ids']).unsqueeze(0).to(model.device)
 att_mask = torch.tensor(sample_df.loc[i, 'attention_mask']).unsqueeze(0).to(model.device)

 with torch.no_grad():
 logits = model(input_ids=input_ids, attention_mask=att_mask).logits.squeeze(0)
 selected = logits[target_ids_tensor] # keep only the 6 target emotions
 pred = int(torch.argmax(selected).item())
 predictions.append(pred)

 del input_ids, att_mask, logits
 if torch.cuda.is_available():
 torch.cuda.empty_cache()

 # Metrics + full classification report
 accuracy = accuracy_score(labels_list, predictions)
 f1 = f1_score(labels_list, predictions, average='weighted', zero_division=0)
 report_dict = classification_report(labels_list, predictions, output_dict=True, zero_division=0)
 report_df = pd.DataFrame(report_dict).transpose().round(4).drop("accuracy", errors="ignore")

 accuracy_row = pd.DataFrame({
 'precision': [""],
 'recall': [""],
 'f1-score': [accuracy],
 'support': [sum(report_df["support"])]
 }, index=["overall_accuracy"])
 final_df = pd.concat([report_df, accuracy_row])

 # Save CSV
 if report_path is None:
 report_path = os.path.join(GOEMOTIONS_PATH, f"class_silencing_class{class_id}.csv")
 os.makedirs(os.path.dirname(report_path), exist_ok=True)

 if experiment_title is None:
 experiment_title = f"GoEmotions per-class silencing (class={class_id}, p={percentage:.2%})"

 if not os.path.exists(report_path):
 with open(report_path, "w") as f:
 f.write(f"# {experiment_title}\n")
 final_df.to_csv(f)
 else:
 with open(report_path, "a") as f:
 f.write(f"\n\n# {experiment_title}\n")
 final_df.to_csv(report_path, mode="a")

 logger.info(f"Accuracy after per-class silencing: {accuracy:.4f}")
 logger.info(f"Weighted F1 Score: {f1:.4f}")
 logger.info(f"Classification report saved to {report_path}")

 # Cleanup
 for h in hook_handles: h.remove()
 logger.info("All hooks removed after evaluation")

 return {"accuracy": accuracy, "f1": f1, "report_df": final_df}

In [ ]:
percentages = SWEEP_PCTS # unified: single source of truth (control panel)

target_class_id = 4 # local 0..5 label in your 6-class slice
report_path = os.path.join(GOEMOTIONS_PATH, f"per_class_silencing_class{target_class_id}.csv")

summary_rows = []

for pct in percentages:
 scores = silence_top_class_percentage_and_evaluate_goemo(
 model=model_goem,
 sample_df=sample_df,
 labels_list=labels_list,
 probe=probe,
 label2idx=label2idx, # local mapping used by the probe
 target_ids_tensor=target_ids_tensor, # original GoEmotions IDs for the 6 emotions
 class_id=target_class_id,
 percentage=pct,
 report_path=report_path,
 experiment_title=f"Per-class silencing (class={target_class_id}, p={pct:.1%})"
 )
 if scores is not None:
 summary_rows.append({
 "percentage": pct,
 "accuracy": scores["accuracy"],
 "f1_weighted": scores["f1"]
 })

# Optional: save a compact summary CSV for plotting
if summary_rows:
 summary_df = pd.DataFrame(summary_rows)
 summary_csv = os.path.join(GOEMOTIONS_PATH, f"per_class_silencing_summary_class{target_class_id}.csv")
 summary_df.to_csv(summary_csv, index=False)
 print(f"Summary saved to {summary_csv}")
else:
 print("No results collected (check logs for errors).")

In [ ]:
# ==== Exact top-k helpers from the probe ====

import numpy as np

def _get_probe_weight_matrix(probe):
 """
 Return W with shape [num_classes, num_features] from the trained probe.
 Supports scikit (coef_) or torch (linear.weight / weight).
 """
 if hasattr(probe, "coef_"): # scikit-learn LogisticRegression
 W = probe.coef_
 elif hasattr(probe, "linear"): # torch module with .linear.weight
 W = probe.linear.weight.detach().cpu().numpy()
 elif hasattr(probe, "weight"): # torch nn.Linear-like
 W = probe.weight.detach().cpu().numpy()
 else:
 raise RuntimeError("Cannot extract weights from probe.")
 W = np.asarray(W)
 if W.ndim == 1:
 W = W[None, :]
 return W # [C, F]

def topk_per_class_exact_from_probe(probe, percentage, class_id):
 """
 Class-conditional ranking: importance = |W[class_id, j]|.
 Returns exactly k = round(p * F) feature indices in [0, F-1].
 """
 W = _get_probe_weight_matrix(probe)
 F = W.shape[1]
 k = int(max(1, round(percentage * F)))
 w = np.abs(W[class_id]) # [F]
 idx = np.argpartition(-w, kth=k-1)[:k]
 idx = idx[np.argsort(-w[idx])] # sort by importance descending
 return idx.tolist()

In [ ]:
def quick_baseline_goemo():
 model_goem.eval()
 preds, trues = [], []
 with torch.no_grad():
 for i, row in sample_df.iterrows():
 input_ids = torch.tensor(row["input_ids"]).unsqueeze(0).to(device_goemo)
 att_mask = torch.tensor(row["attention_mask"]).unsqueeze(0).to(device_goemo)
 logits = model_goem(input_ids=input_ids, attention_mask=att_mask).logits.squeeze(0)
 sel_logits = logits[target_ids_tensor]
 preds.append(int(torch.argmax(sel_logits)))
 trues.append(labels_list[i])
 from sklearn.metrics import accuracy_score, f1_score
 return accuracy_score(trues, preds), f1_score(trues, preds, average="weighted")

acc0, f10 = quick_baseline_goemo()
print(f"Baseline — Acc={acc0:.4f} F1w={f10:.4f}")

In [ ]:
# ==== Per-class silencing (exact k) ====

import os, logging, torch, pandas as pd
from collections import defaultdict
from sklearn.metrics import accuracy_score, f1_score, classification_report

logger = logging.getLogger(__name__)

def per_class_silencing_goemo(
 class_id,
 percentage=0.10, # e.g., 0.10 = 10%
 layers_scope="all", # "all"or "last_only"
 report_path=None,
 experiment_title=None
):
 """
 Silence top-k neurons for a given local class_id using exact top-k from the probe.
 Uses: model_goem, probe, label2idx, sample_df, labels_list, target_ids_tensor, device_goemo, GOEMOTIONS_PATH.
 """

 # 1) Exact top-k indices (global indices over all layers)
 topk_global = topk_per_class_exact_from_probe(probe, percentage=percentage, class_id=class_id)

 H = model_goem.config.hidden_size
 L = model_goem.config.num_hidden_layers
 F = H * L
 k_teor = int(round(percentage * F))
 k_real = len(topk_global)
 logger.info(f"[PerClassSilencing] class={class_id} p={percentage:.0%} k_teor={k_teor} k_real={k_real} ratio={k_real/F:.2%}")

 if layers_scope not in ("all", "last_only"):
 raise ValueError("layers_scope must be 'all' or 'last_only'")

 if layers_scope == "last_only":
 last = L - 1
 topk_global = [g for g in topk_global if last*H <= g < (last+1)*H]

 if len(topk_global) == 0:
 logger.warning("[PerClassSilencing] No neurons selected for this class/percentage after scope filter.")
 return None

 # 2) Map global indices -> {layer: [local dims]}
 layer_to_indices = defaultdict(list)
 for g in topk_global:
 Lidx = g // H
 didx = g % H
 if 0 <= Lidx < L:
 layer_to_indices[Lidx].append(int(didx))

 # 3) Register forward hooks (silence CLS dims at encoder outputs)
 handles = []
 for Lidx, idxs in layer_to_indices.items():
 if not idxs:
 continue
 h = model_goem.bert.encoder.layer[Lidx].output.LayerNorm.register_forward_hook(
 make_cls_silence_hook(idxs)
 )
 handles.append(h)

 # 4) Inference on the 6-way slice
 model_goem.eval()
 preds, trues = [], []
 with torch.no_grad():
 for i, row in sample_df.iterrows():
 input_ids = torch.tensor(row["input_ids"]).unsqueeze(0).to(device_goemo)
 att_mask = torch.tensor(row["attention_mask"]).unsqueeze(0).to(device_goemo)

 logits = model_goem(input_ids=input_ids, attention_mask=att_mask).logits.squeeze(0)
 sel_logits = logits[target_ids_tensor] # keep only the 6 target emotions
 pred_local = int(torch.argmax(sel_logits).item())

 preds.append(pred_local)
 trues.append(labels_list[i])

 # 5) Metrics + full per-class report
 accuracy = accuracy_score(trues, preds)
 f1w = f1_score(trues, preds, average='weighted', zero_division=0)
 report_dict = classification_report(trues, preds, output_dict=True, zero_division=0)
 report_df = pd.DataFrame(report_dict).transpose().round(4)
 report_df = report_df.drop("accuracy", errors="ignore")

 accuracy_row = pd.DataFrame({
 'precision': [""],
 'recall': [""],
 'f1-score': [accuracy],
 'support': [sum(report_df["support"])]
 }, index=["overall_accuracy"])
 final_df = pd.concat([report_df, accuracy_row])

 # 6) Save CSV
 if report_path is None:
 report_path = os.path.join(GOEMOTIONS_PATH, f"classification_report_per_class_silencing_class{class_id}.csv")
 os.makedirs(os.path.dirname(report_path), exist_ok=True)

 if experiment_title is None:
 experiment_title = f"Per-class silencing (class={class_id}, p={percentage:.0%}, scope={layers_scope})"

 if not os.path.exists(report_path):
 with open(report_path, "w") as f:
 f.write(f"# {experiment_title}\n")
 final_df.to_csv(f)
 else:
 with open(report_path, "a") as f:
 f.write(f"\n\n# {experiment_title}\n")
 final_df.to_csv(report_path, mode="a")

 logger.info(f"Accuracy after per-class silencing: {accuracy:.4f}")
 logger.info(f"Weighted F1 Score: {f1w:.4f}")
 logger.info(f"Classification report saved to {report_path}")

 # 7) Cleanup
 for h in handles:
 h.remove()
 logger.info("All hooks removed after evaluation")

 return {
 "accuracy": accuracy,
 "f1": f1w,
 "k_teor": k_teor,
 "k_real": k_real,
 "report_df": final_df
 }

In [ ]:
# ==== Example: iterative sweep (same style you used) ====

percentages = SWEEP_PCTS # unified: single source of truth (control panel)

target_class_id = 4 # local id in {0..5}

for pct in percentages:
 per_class_silencing_goemo(
 class_id=target_class_id,
 percentage=pct,
 layers_scope="all",
 report_path=os.path.join(GOEMOTIONS_PATH, f"per_class_silencing_class{target_class_id}.csv"),
 experiment_title=f"Per-class silencing (class={target_class_id}, p={pct:.0%}, scope=all)"
 )

In [ ]:
# ========= Per-class silencing + summary row + colorblind plot =========
import os, logging, numpy as np, torch, pandas as pd
from collections import defaultdict
from sklearn.metrics import accuracy_score, f1_score, classification_report
import matplotlib.pyplot as plt

logger = logging.getLogger(__name__)

# Okabe–Ito colorblind-safe palette (7 colores; 6 clases + weighted)
OKABE_ITO = ["#0072B2","#D55E00","#009E73","#CC79A7","#F0E442","#56B4E9","#000000"]

def _get_probe_W(probe):
 # Robust extraction of weight matrix [C, F]
 if hasattr(probe, "coef_"):
 W = probe.coef_
 elif hasattr(probe, "linear"):
 W = probe.linear.weight.detach().cpu().numpy()
 elif hasattr(probe, "weight"):
 W = probe.weight.detach().cpu().numpy()
 else:
 raise RuntimeError("Cannot extract probe weights.")
 W = np.asarray(W)
 if W.ndim == 1: W = W[None, :]
 return W

def _topk_per_class_exact(probe, percentage, class_id):
 # Exact top-k by absolute weight for the class
 W = _get_probe_W(probe) # [C, F]
 F = W.shape[1]
 k = int(max(1, round(percentage * F)))
 w = np.abs(W[class_id]) # [F]
 idx = np.argpartition(-w, kth=k-1)[:k]
 idx = idx[np.argsort(-w[idx])]
 return idx.tolist()

def _map_global_to_layer_indices(model, global_indices, layers_scope="all"):
 # Map flat indices -> {layer: [local_dims]}
 H = model.config.hidden_size
 L = model.config.num_hidden_layers
 if layers_scope not in ("all", "last_only"):
 raise ValueError("layers_scope must be 'all' or 'last_only'")
 if layers_scope == "last_only":
 last = L - 1
 global_indices = [g for g in global_indices if last*H <= g < (last+1)*H]
 layer_to_indices = defaultdict(list)
 for g in global_indices:
 Li = g // H
 dj = g % H
 if 0 <= Li < L:
 layer_to_indices[Li].append(int(dj))
 return layer_to_indices

def _append_summary_row(report_dict, percentage, summary_path, class_names=None):
 """
 Append one compact row to the summary CSV with a fixed schema.
 If the file exists, reindex row to match existing header.
 """
 import numpy as np, os, pandas as pd

 # per-class F1
 per_class = {k: v.get("f1-score", np.nan)
 for k, v in report_dict.items()
 if isinstance(v, dict) and k.isdigit()}

 weighted_f1 = report_dict.get("weighted avg", {}).get("f1-score", np.nan)
 accuracy = report_dict.get("accuracy", np.nan)

 # build class columns in a stable order
 if class_names:
 class_cols = {f"f1_{name}": per_class.get(str(i), np.nan)
 for i, name in enumerate(class_names)}
 else:
 # fallback: numeric names if none provided
 class_cols = {f"f1_class_{i}": per_class.get(str(i), np.nan)
 for i in sorted(map(int, per_class.keys()))}

 row = {"percentage": percentage, "f1_weighted": weighted_f1, "accuracy": accuracy, **class_cols}
 df_row = pd.DataFrame([row])

 os.makedirs(os.path.dirname(summary_path), exist_ok=True)

 if os.path.exists(summary_path):
 # read only header to get existing schema
 header_cols = pd.read_csv(summary_path, nrows=0, comment="#").columns.tolist()
 # align new row to existing columns (fill missing with NaN)
 df_row = df_row.reindex(columns=header_cols, fill_value=np.nan)
 df_row.to_csv(summary_path, mode="a", header=False, index=False)
 else:
 # first time write with header using current schema
 df_row.to_csv(summary_path, index=False)

def plot_goemo_f1_curves(summary_csv, title=None, save_path=None, class_names=None):
 """
 Plot per-class F1 and weighted F1 vs. percentage from a robust summary CSV.
 Ignores comment lines and handles stray blank lines.
 """
 import pandas as pd
 import matplotlib.pyplot as plt
 from cycler import cycler

 # robust read: ignore lines starting with '#'
 if not os.path.exists(summary_csv):
 print(f"[plot] skipped (missing file): {summary_csv}")
 return
 df = pd.read_csv(summary_csv, comment="#")
 df = df.dropna(how="all") # drop empty lines
 df = df.sort_values("percentage")

 # choose class columns
 if class_names:
 class_cols = [f"f1_{name}"for name in class_names if f"f1_{name}"in df.columns]
 else:
 class_cols = [c for c in df.columns if c.startswith("f1_class_")]

 # colorblind-friendly palette (Okabe–Ito)
 OKABE_ITO = ["#0072B2","#D55E00","#009E73","#CC79A7","#F0E442","#56B4E9","#000000"]
 plt.figure(figsize=(8,5))
 plt.gca().set_prop_cycle(cycler(color=OKABE_ITO[:len(class_cols)+1]))

 for col in class_cols:
 plt.plot(df["percentage"]*100.0, df[col], marker="o", linewidth=2, label=col)

 if "f1_weighted"in df.columns:
 plt.plot(df["percentage"]*100.0, df["f1_weighted"], marker="o",
 linewidth=3, linestyle="--", label="f1_weighted")

 plt.xlabel("Silenced neurons (%)")
 plt.ylabel("F1-score")
 if title: plt.title(title)
 plt.ylim(0.0, 1.0)
 plt.grid(True, alpha=0.3)
 plt.legend(loc="best")
 plt.tight_layout()
 if save_path: plt.savefig(save_path, dpi=200)
 plt.show()
 plt.close()

 
def per_class_silencing_goemo(
 class_id,
 percentage=0.10,
 layers_scope="all",
 report_path=None,
 experiment_title=None,
 summary_path=None, # NEW: CSV with per-class F1 rows
 class_names=None # e.g., TARGET_EMOTIONS order for local IDs
):
 """
 Silence top-k neurons for a target class and:
 - save full classification_report CSV,
 - append a compact per-class F1 row to 'summary_path'.
 Requires: model_goem, probe, label2idx, sample_df, labels_list, target_ids_tensor, device_goemo.
 """
 # 1) Select exact top-k global indices
 topk_global = _topk_per_class_exact(probe, percentage=percentage, class_id=class_id)

 H = model_goem.config.hidden_size
 L = model_goem.config.num_hidden_layers
 F = H * L
 k_teor = int(round(percentage * F))
 k_real = len(topk_global)
 logger.info(f"[PerClassSilencing] class={class_id} p={percentage:.2%} k_teor={k_teor} k_real={k_real} ratio={100*k_real/F:.2f}%")

 # 2) Map to layers and register hooks at encoder.layer[i].output
 layer_to_indices = _map_global_to_layer_indices(model_goem, topk_global, layers_scope=layers_scope)
 handles = []
 enc = model_goem.bert.encoder.layer
 for Li, idxs in layer_to_indices.items():
 if not idxs: continue
 h = enc[Li].output.register_forward_hook(make_cls_silence_hook(idxs))
 handles.append(h)

 # 3) Inference (6-way slice)
 model_goem.eval()
 preds, trues = [], []
 with torch.no_grad():
 for i, row in sample_df.iterrows():
 input_ids = torch.tensor(row["input_ids"]).unsqueeze(0).to(device_goemo)
 att_mask = torch.tensor(row["attention_mask"]).unsqueeze(0).to(device_goemo)
 logits = model_goem(input_ids=input_ids, attention_mask=att_mask).logits.squeeze(0)
 sel_logits = logits[target_ids_tensor]
 preds.append(int(torch.argmax(sel_logits)))
 trues.append(labels_list[i])

 # 4) Metrics + full report
 acc = accuracy_score(trues, preds)
 f1w = f1_score(trues, preds, average='weighted', zero_division=0)
 report_dict = classification_report(trues, preds, output_dict=True, zero_division=0)
 report_df = pd.DataFrame(report_dict).transpose().round(4)
 report_df = report_df.drop("accuracy", errors="ignore")

 accuracy_row = pd.DataFrame({
 'precision': [""],
 'recall': [""],
 'f1-score': [acc],
 'support': [sum(report_df["support"])]
 }, index=["overall_accuracy"])
 final_df = pd.concat([report_df, accuracy_row])

 # 5) Save full report
 if report_path is None:
 report_path = os.path.join(GOEMOTIONS_PATH, f"per_class_silencing_class{class_id}.csv")
 os.makedirs(os.path.dirname(report_path), exist_ok=True)
 if experiment_title is None:
 experiment_title = f"Per-class silencing (class={class_id}, p={percentage:.2%}, scope={layers_scope})"
 if not os.path.exists(report_path):
 with open(report_path, "w") as f:
 f.write(f"# {experiment_title}\n")
 final_df.to_csv(f)
 else:
 with open(report_path, "a") as f:
 f.write(f"\n\n# {experiment_title}\n")
 final_df.to_csv(report_path, mode="a")

 # 6) Append compact summary row
 if summary_path:
 _append_summary_row(report_dict, percentage, summary_path, class_names=class_names)

 logger.info(f"Acc: {acc:.4f} F1w: {f1w:.4f} saved report & summary")

 # 7) Cleanup
 for h in handles: h.remove()
 return {"accuracy": acc, "f1": f1w, "k_teor": k_teor, "k_real": k_real}

In [ ]:
# Start fresh: remove any old summary with mixed schema
if os.path.exists(summary_csv):
 os.remove(summary_csv)

# Optional: pre-create a fixed header so all rows align
fixed_cols = ["percentage","f1_weighted","accuracy"] + [f"f1_{name}"for name in TARGET_EMOTIONS]
pd.DataFrame(columns=fixed_cols).to_csv(summary_csv, index=False)

In [ ]:
# --- sanity baseline (optional, to detect stale hooks) ---
def quick_baseline_goemo():
 model_goem.eval()
 preds, trues = [], []
 with torch.no_grad():
 for i, row in sample_df.iterrows():
 input_ids = torch.tensor(row["input_ids"]).unsqueeze(0).to(device_goemo)
 att_mask = torch.tensor(row["attention_mask"]).unsqueeze(0).to(device_goemo)
 logits = model_goem(input_ids=input_ids, attention_mask=att_mask).logits.squeeze(0)
 sel_logits = logits[target_ids_tensor]
 preds.append(int(torch.argmax(sel_logits)))
 trues.append(labels_list[i])
 return (accuracy_score(trues, preds),
 f1_score(trues, preds, average="weighted", zero_division=0))

acc0, f10 = quick_baseline_goemo()
logger.info(f"[Baseline] Acc={acc0:.4f} F1w={f10:.4f}")

# --- sweep and plot ---
percentages = SWEEP_PCTS # unified: single source of truth (control panel)

target_class_id = 4 # local 0..5
summary_csv = os.path.join(GOEMOTIONS_PATH, f"per_class_silencing_summary_class{target_class_id}.csv")
report_csv = os.path.join(GOEMOTIONS_PATH, f"per_class_silencing_class{target_class_id}.csv")

# Optional: start summary with baseline row at p=0.0
if os.path.exists(summary_csv):
 os.remove(summary_csv)
baseline_report = classification_report([0], [0], output_dict=True) # dummy structure
baseline_report["weighted avg"]["f1-score"] = f10
baseline_report["accuracy"] = acc0
# put NaN for per-class f1s
for k in list(baseline_report.keys()):
 if isinstance(baseline_report[k], dict) and str(k).isdigit():
 baseline_report[k]["f1-score"] = np.nan
_append_summary_row(baseline_report, 0.0, summary_csv, class_names=TARGET_EMOTIONS)

for pct in percentages:
 per_class_silencing_goemo(
 class_id=target_class_id,
 percentage=pct,
 layers_scope="all",
 report_path=report_csv,
 experiment_title=f"Per-class silencing (class={target_class_id}, p={pct:.1%}, scope=all)",
 summary_path=summary_csv,
 class_names=TARGET_EMOTIONS
 )

plot_goemo_f1_curves(
 summary_csv,
 title=f"Per-class F1 vs. % silenced — class {target_class_id}",
 save_path=os.path.join(GOEMOTIONS_PATH, f"per_class_silencing_class{target_class_id}.png"),
 class_names=TARGET_EMOTIONS
)

In [ ]:
import os, json, logging, numpy as np, pandas as pd, torch
from collections import defaultdict
from sklearn.metrics import accuracy_score, f1_score, classification_report
import matplotlib.pyplot as plt

logger = logging.getLogger(__name__)

# --- Hook to zero selected CLS dimensions ---
def make_cls_silence_hook(indices: list[int]):
 idx = torch.tensor(indices, dtype=torch.long) if indices else torch.tensor([], dtype=torch.long)
 def hook(module, _inp, out):
 if not isinstance(out, torch.Tensor): 
 return out
 o = out.clone()
 if o.dim() == 3: # [B, T, H]
 cls = o[:, 0, :]
 if idx.numel() > 0:
 cls[:, idx.to(o.device)] = 0.0
 o[:, 0, :] = cls
 elif o.dim() == 2: # [B, H]
 if idx.numel() > 0:
 o[:, idx.to(o.device)] = 0.0
 return o
 return hook

# --- Map global neuron indices -> {layer: [local_dims]} ---
def _map_global_to_layer_indices(model, global_indices, layers_scope="all"):
 H = model.config.hidden_size
 L = model.config.num_hidden_layers
 if layers_scope not in ("all", "last_only"):
 raise ValueError("layers_scope must be 'all' or 'last_only'")
 if layers_scope == "last_only":
 last = L - 1
 global_indices = [g for g in global_indices if last*H <= g < (last+1)*H]

 layer_to_indices = defaultdict(list)
 for g in global_indices:
 layer = g // H
 dim = g % H
 if 0 <= layer < L:
 layer_to_indices[layer].append(int(dim))
 return layer_to_indices

# --- Robust top-k per class from probe (tries NeuroX helper, else coef_ fallback) ---
def top_neurons_for_class_goemo(probe, class_id, percentage: float, label2idx: dict):
 try:
 # NeuroX helper returns (global_top, per_class_dict_or_array)
 _, per_class = linear_probe.get_top_neurons(probe, percentage=percentage, class_to_idx=label2idx)
 arr = per_class[class_id] if isinstance(per_class, dict) else per_class[class_id]
 return [int(x) for x in np.asarray(arr).ravel().tolist()]
 except Exception:
 # Fallback: extract coef_ and rank by |weight|
 W = None
 for attr in ("coef_", "coef", "weights", "W", "weight"):
 if hasattr(probe, attr):
 W = getattr(probe, attr); break
 if isinstance(probe, dict) and attr in probe:
 W = probe[attr]; break
 if W is None:
 raise RuntimeError("Cannot extract weights from probe for fallback.")
 W = np.asarray(W)
 if W.ndim == 1: W = W[None, :]
 cidx = label2idx.get(class_id, class_id)
 vec = np.abs(W[cidx])
 k = max(1, round(percentage * vec.shape[0]))
 idxs = np.argpartition(-vec, k-1)[:k]
 idxs = idxs[np.argsort(-vec[idxs])]
 return idxs.astype(int).tolist()

# --- Baseline on the 6-way slice ---
def quick_baseline_goemo():
 preds, trues = [], []
 model_goem.eval()
 with torch.no_grad():
 for i, row in sample_df.iterrows():
 input_ids = torch.tensor(row["input_ids"]).unsqueeze(0).to(device_goemo)
 att_mask = torch.tensor(row["attention_mask"]).unsqueeze(0).to(device_goemo)
 logits = model_goem(input_ids=input_ids, attention_mask=att_mask).logits.squeeze(0)
 sel_logits = logits[target_ids_tensor]
 preds.append(int(torch.argmax(sel_logits)))
 trues.append(labels_list[i])
 acc = accuracy_score(trues, preds)
 f1w = f1_score(trues, preds, average='weighted', zero_division=0)
 return acc, f1w

# --- Summary CSV helpers (consistent schema) ---
def init_goemo_summary(summary_path, class_names):
 cols = ["percentage", "f1_weighted", "accuracy"] + [f"f1_{n}"for n in class_names]
 os.makedirs(os.path.dirname(summary_path), exist_ok=True)
 if os.path.exists(summary_path):
 os.remove(summary_path)
 pd.DataFrame(columns=cols).to_csv(summary_path, index=False)

def append_goemo_summary_row(summary_path, percentage, f1_weighted, accuracy, per_class_f1, class_names):
 if per_class_f1 is None:
 per_class_f1 = [np.nan]*len(class_names)
 row = {"percentage": percentage, "f1_weighted": f1_weighted, "accuracy": accuracy}
 row.update({f"f1_{name}": per_class_f1[i] for i, name in enumerate(class_names)})
 cols = ["percentage","f1_weighted","accuracy"] + [f"f1_{n}"for n in class_names]
 pd.DataFrame([row], columns=cols).to_csv(summary_path, mode="a", header=False, index=False)

def extract_per_class_f1(report_dict, num_classes):
 out = []
 for i in range(num_classes):
 key = str(i)
 f1 = np.nan
 if key in report_dict and isinstance(report_dict[key], dict):
 f1 = report_dict[key].get("f1-score", np.nan)
 out.append(f1)
 return out

# --- Plot (Matplotlib, colorblind-friendly) ---
def plot_goemo_f1_curves(summary_csv, title=None, save_path=None, class_names=None):
 if not os.path.exists(summary_csv):
 print(f"[plot] skipped (missing file): {summary_csv}")
 return
 df = pd.read_csv(summary_csv).sort_values("percentage")
 plt.style.use("tableau-colorblind10")
 fig, ax = plt.subplots(figsize=(7,4))
 # per-class
 for name in (class_names or []):
 ax.plot(df["percentage"]*100, df[f"f1_{name}"], marker="o", linewidth=1.8, label=f"F1 {name}")
 # weighted
 ax.plot(df["percentage"]*100, df["f1_weighted"], marker="o", linewidth=2.2, label="F1 weighted", linestyle="--")
 ax.set_xlabel("% neurons silenced (class-specific)")
 ax.set_ylabel("F1-score")
 if title: ax.set_title(title)
 ax.grid(True, alpha=0.3)
 ax.legend(ncol=2, fontsize=8)
 fig.tight_layout()
 if save_path:
 fig.savefig(save_path, dpi=200, bbox_inches="tight")
 plt.show()

In [ ]:
from typing import Optional, List

def per_class_silencing_goemo(
 class_id: int,
 percentage: float = 0.10,
 layers_scope: str = "all",
 report_path: Optional[str] = None,
 experiment_title: Optional[str] = None,
 summary_path: Optional[str] = None,
 class_names: Optional[List[str]] = None
):
 # 1) Select top-k global indices for the target class
 topk_global = top_neurons_for_class_goemo(probe, class_id, percentage, label2idx)
 if len(topk_global) == 0:
 logger.warning("[PerClassSilencing] No neurons selected.")
 return None

 # 2) Map to per-layer lists and register hooks
 layer_to_indices = _map_global_to_layer_indices(model_goem, topk_global, layers_scope=layers_scope)
 handles = []
 try:
 for L, idxs in layer_to_indices.items():
 if not idxs: 
 continue
 h = model_goem.bert.encoder.layer[L].output.LayerNorm.register_forward_hook(
 make_cls_silence_hook(idxs)
 )
 handles.append(h)

 # 3) Inference on the 6-way slice
 model_goem.eval()
 preds, trues = [], []
 with torch.no_grad():
 for i, row in sample_df.iterrows():
 input_ids = torch.tensor(row["input_ids"]).unsqueeze(0).to(device_goemo)
 att_mask = torch.tensor(row["attention_mask"]).unsqueeze(0).to(device_goemo)
 logits = model_goem(input_ids=input_ids, attention_mask=att_mask).logits.squeeze(0)
 sel_logits = logits[target_ids_tensor]
 preds.append(int(torch.argmax(sel_logits)))
 trues.append(labels_list[i])

 acc = accuracy_score(trues, preds)
 f1w = f1_score(trues, preds, average='weighted', zero_division=0)

 rep = classification_report(
 trues, preds,
 labels=list(range(len(TARGET_EMOTIONS))),
 target_names=[str(i) for i in range(len(TARGET_EMOTIONS))],
 output_dict=True, zero_division=0
 )
 df = pd.DataFrame(rep).transpose().round(4).drop("accuracy", errors="ignore")
 acc_row = pd.DataFrame({'precision': [""],'recall': [""],'f1-score': [acc],'support': [sum(df["support"])]},
 index=["overall_accuracy"])
 final_df = pd.concat([df, acc_row])

 logger.info(f"[PerClassSilencing] class={class_id} p={percentage:.0%} scope={layers_scope} "
 f"Acc={acc:.4f} F1w={f1w:.4f}")

 # 4) Save detailed report
 if report_path is None:
 report_path = os.path.join(GOEMOTIONS_PATH, f"class_silencing_class{class_id}.csv")
 os.makedirs(os.path.dirname(report_path), exist_ok=True)
 if experiment_title is None:
 experiment_title = f"Per-class silencing (class={class_id}, p={percentage:.1%}, scope={layers_scope})"
 if not os.path.exists(report_path):
 with open(report_path, "w") as f:
 f.write(f"# {experiment_title}\n")
 final_df.to_csv(f)
 else:
 with open(report_path, "a") as f:
 f.write(f"\n\n# {experiment_title}\n")
 final_df.to_csv(report_path, mode="a")

 # 5) Append summary row (optional)
 if summary_path is not None and class_names is not None:
 per_cls_f1 = extract_per_class_f1(rep, num_classes=len(class_names))
 append_goemo_summary_row(summary_path, percentage, f1w, acc, per_cls_f1, class_names)

 return {"accuracy": acc, "f1": f1w, "report_df": final_df}

 finally:
 for h in handles:
 h.remove()

In [ ]:
# Define sweep and outputs
percentages = [0.0] + SWEEP_PCTS # baseline + unified sweep
target_class_id = 1 # local id (0..5)

summary_csv = os.path.join(GOEMOTIONS_PATH, f"per_class_silencing_summary_class{target_class_id}.csv")
report_csv = os.path.join(GOEMOTIONS_PATH, f"per_class_silencing_class{target_class_id}.csv")

# Fresh summary with consistent header
init_goemo_summary(summary_csv, TARGET_EMOTIONS)

# Optional: baseline row at p=0 if not included above
if percentages[0] != 0.0:
 acc0, f10 = quick_baseline_goemo()
 append_goemo_summary_row(summary_csv, 0.0, f10, acc0, per_class_f1=None, class_names=TARGET_EMOTIONS)

# Run sweep (writes detailed report + appends to summary)
for pct in percentages:
 per_class_silencing_goemo(
 class_id=target_class_id,
 percentage=pct,
 layers_scope="all",
 report_path=report_csv,
 experiment_title=f"Per-class silencing (class={target_class_id}, p={pct:.1%}, scope=all)",
 summary_path=summary_csv,
 class_names=TARGET_EMOTIONS
 )

# Plot curves (colorblind-friendly)
emotion_name = TARGET_EMOTIONS[target_class_id]

plot_goemo_f1_curves(
 summary_csv,
 title=f"Per-class F1 vs. % silenced — {emotion_name}",
 save_path=os.path.join(GOEMOTIONS_PATH, f"per_class_silencing_{emotion_name}.png"),
 class_names=TARGET_EMOTIONS
)

#### Second try

In [ ]:
import numpy as np
import torch

def get_top_k_neurons_for_class_exact_goemo(probe, percentage, class_to_idx, class_id):
 """
 Return top-k neurons most important for a specific class, measured by absolute weight.
 Works with either a torch probe (probe.linear.weight) or a sklearn probe (probe.coef_).
 Output: list[int] of global neuron indices in [0, H*L).
 """
 # Try torch-style probe first
 if hasattr(probe, "linear") and hasattr(probe.linear, "weight"):
 W = probe.linear.weight.detach().abs().cpu().numpy() # [num_classes, num_neurons]
 # Fallback: sklearn-style (coef_)
 elif hasattr(probe, "coef_"):
 W = np.abs(np.asarray(probe.coef_)) # [num_classes, num_neurons]
 if W.ndim == 1:
 W = W[None, :]
 else:
 raise RuntimeError("Probe weights not found (expected .linear.weight or .coef_).")

 # Map class_id if needed (here class_id ya es local 0..5, normalmente no hace falta)
 cidx = class_to_idx.get(class_id, class_id)

 class_weights = W[cidx] # [num_neurons]
 total_neurons = class_weights.shape[0]
 top_n = max(1, round(percentage * total_neurons))

 # índices de mayor |peso|
 top_indices = class_weights.argsort()[-top_n:]
 return top_indices.tolist()

In [ ]:
import os, json, logging, pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report

logger = logging.getLogger(__name__)

def get_encoder_layers(model):
 if hasattr(model, "bert"):
 return model.bert.encoder.layer
 elif hasattr(model, "longformer"):
 return model.longformer.encoder.layer
 elif hasattr(model, "distilbert"):
 return model.distilbert.transformer.layer
 else:
 raise NotImplementedError("Unsupported model architecture.")

def silence_top_class_percentage_and_evaluate_goemo(
 model,
 sample_df,
 labels_list,
 probe,
 label2idx,
 class_id,
 percentage=0.10,
 report_path=None,
 experiment_title=None
):
 """
 Silences the top-k neurons for the given class (global indices across ALL layers),
 evaluates on the 6-way GoEmotions slice, and writes a full per-class report.
 Uses your existing: target_ids_tensor, device_goemo, make_cls_silence_hook.
 """
 hidden_dim = model.config.hidden_size
 num_layers = model.config.num_hidden_layers

 # 1) Top-k neurons per class (global indices 0..H*L-1)
 top_class_neurons = get_top_k_neurons_for_class_exact_goemo(
 probe, percentage=percentage, class_to_idx=label2idx, class_id=class_id
 )
 logger.info(f"Silencing {len(top_class_neurons)} neurons for class {class_id} ({percentage:.2%} of global)")

 # 2) Guardar índices (opcional, como en malware)
 neurons_dir = os.path.join(GOEMOTIONS_PATH, "neurons")
 os.makedirs(neurons_dir, exist_ok=True)
 json_path = os.path.join(neurons_dir, f"top_{int(percentage*100)}p_neurons_class_{class_id}.json")
 with open(json_path, "w") as f:
 json.dump(top_class_neurons, f, indent=2)
 logger.info(f"Saved neuron indices to {json_path}")

 # 3) Registrar hooks por capa (mismo mapeo que malware)
 encoder_layers = get_encoder_layers(model)
 hook_handles = []
 for i in range(num_layers):
 indices_layer = [idx - i * hidden_dim for idx in top_class_neurons
 if i * hidden_dim <= idx < (i + 1) * hidden_dim]
 if indices_layer:
 logger.info(f"Layer {i}: silencing {len(indices_layer)} neurons for class {class_id}")
 handle = encoder_layers[i].output.register_forward_hook(make_cls_silence_hook(indices_layer))
 hook_handles.append(handle)

 # 4) Inferencia (seleccionando solo los 6 logits de interés)
 model.eval()
 predictions = []
 for i in range(len(sample_df)):
 input_ids_tensor = torch.tensor(sample_df.loc[i, 'input_ids']).unsqueeze(0).to(model.device)
 attention_mask_tensor = torch.tensor(sample_df.loc[i, 'attention_mask']).unsqueeze(0).to(model.device)

 with torch.no_grad():
 logits = model(input_ids=input_ids_tensor, attention_mask=attention_mask_tensor).logits.squeeze(0)
 sel_logits = logits[target_ids_tensor] # <- usa tu mapeo 6-way
 pred = int(torch.argmax(sel_logits).item())
 predictions.append(pred)

 del input_ids_tensor, attention_mask_tensor, logits
 if model.device.type == "cuda":
 torch.cuda.empty_cache()

 # 5) Métricas + reporte (mismo patrón que malware)
 accuracy = accuracy_score(labels_list, predictions)
 f1 = f1_score(labels_list, predictions, average='weighted', zero_division=0)
 report_dict = classification_report(labels_list, predictions, output_dict=True, zero_division=0)
 report_df = pd.DataFrame(report_dict).transpose().round(4)
 report_df = report_df.drop("accuracy", errors="ignore")

 accuracy_row = pd.DataFrame({
 'precision': [""],
 'recall': [""],
 'f1-score': [accuracy],
 'support': [sum(report_df["support"])]
 }, index=["overall_accuracy"])
 final_df = pd.concat([report_df, accuracy_row])

 if report_path is None:
 report_path = os.path.join(GOEMOTIONS_PATH, f"per_class_silencing_class{class_id}.csv")
 os.makedirs(os.path.dirname(report_path), exist_ok=True)

 if experiment_title is None:
 experiment_title = f"Silencing top {percentage:.2%} neurons for class {class_id}"

 if not os.path.exists(report_path):
 with open(report_path, "w") as f:
 f.write(f"# {experiment_title}\n")
 final_df.to_csv(f)
 else:
 with open(report_path, "a") as f:
 f.write(f"\n\n# {experiment_title}\n")
 final_df.to_csv(report_path, mode="a")

 logger.info(f"Accuracy after class-specific silencing: {accuracy:.4f}")
 logger.info(f"Weighted F1 Score: {f1:.4f}")
 logger.info(f"Classification report saved to {report_path}")

 # 6) Quitar hooks
 for handle in hook_handles:
 handle.remove()
 logger.info("All hooks removed after evaluation")

In [ ]:
# Ejemplo: barrido de porcentajes para la clase local 4
percentages = SWEEP_PCTS # unified: single source of truth (control panel)

target_class_id = 3 # 0..5 en tu slice local

for pct in percentages:
 silence_top_class_percentage_and_evaluate_goemo(
 model=model_goem,
 sample_df=sample_df,
 labels_list=labels_list,
 probe=probe,
 label2idx=label2idx,
 class_id=target_class_id,
 percentage=pct,
 experiment_title=f"Silencing {pct*100:.1f}% of Neurons for Class {target_class_id}",
 report_path=os.path.join(GOEMOTIONS_PATH, f"per_class_silencing_class{target_class_id}.csv")
 )

#### Third try

In [ ]:
# Must already exist: probe, idx2label (del entrenamiento del probe), TARGET_EMOTIONS, target_ids_tensor

# local (0..5) -> probe row index
local2probe = {v: k for k, v in idx2label.items()}

print("Probe idx2label (idx -> local_id):", idx2label)
print("local2probe (local_id -> probe_row):", local2probe)
print("TARGET_EMOTIONS:", TARGET_EMOTIONS)
print("target_ids_tensor:", target_ids_tensor.tolist())

In [ ]:
import numpy as np

def get_top_k_neurons_for_class_exact_goemo(probe, percentage: float, probe_class_idx: int) -> list[int]:
 # Works with torch probe (probe.linear.weight) or sklearn (probe.coef_)
 if hasattr(probe, "linear") and hasattr(probe.linear, "weight"):
 W = probe.linear.weight.detach().abs().cpu().numpy() # [C, F]
 elif hasattr(probe, "coef_"):
 W = np.abs(np.asarray(probe.coef_)) # [C, F] or [F]
 if W.ndim == 1:
 W = W[None, :]
 else:
 raise RuntimeError("Probe weights not found (.linear.weight or .coef_).")

 class_weights = W[int(probe_class_idx)] # [F]
 F = class_weights.shape[0]
 top_n = max(1, int(np.floor(percentage * F)))
 idx = np.argpartition(class_weights, -top_n)[-top_n:]
 idx = idx[np.argsort(-class_weights[idx])] # sort by magnitude desc
 return idx.tolist()

In [ ]:
import os, json, torch, pandas as pd, logging
from sklearn.metrics import accuracy_score, f1_score, classification_report

logger = logging.getLogger(__name__)

def get_encoder_layers(model):
 if hasattr(model, "bert"): return model.bert.encoder.layer
 if hasattr(model, "longformer"): return model.longformer.encoder.layer
 if hasattr(model, "distilbert"): return model.distilbert.transformer.layer
 raise NotImplementedError("Unsupported model architecture.")

def silence_top_class_percentage_and_evaluate_goemo(
 model,
 sample_df,
 labels_list,
 class_id_local: int,
 percentage: float,
 report_path: str = None,
 experiment_title: str = None
):
 # Map local class (0..5) -> probe row
 probe_row = int(local2probe[int(class_id_local)])

 # 1) Select global indices (0..H*L-1)
 top_global = get_top_k_neurons_for_class_exact_goemo(probe, percentage, probe_row)
 hidden_dim = model.config.hidden_size
 num_layers = model.config.num_hidden_layers
 total_neurons = hidden_dim * num_layers
 k_teor = int(np.floor(percentage * total_neurons))
 k_real = len(top_global)
 logger.info(f"[PerClassSilencing] class={class_id_local} p={percentage:.0%} "
 f"k_teor={k_teor} k_real={k_real} ratio={k_real/total_neurons:.2%}")

 # 2) Register hooks per layer (CLS dims only)
 enc = get_encoder_layers(model)
 handles = []
 for L in range(num_layers):
 idxs = [g - L*hidden_dim for g in top_global if L*hidden_dim <= g < (L+1)*hidden_dim]
 if idxs:
 h = enc[L].output.register_forward_hook(make_cls_silence_hook(idxs))
 handles.append(h)

 # 3) Inference on 6-way slice
 model.eval()
 preds, trues = [], []
 with torch.no_grad():
 for i, row in sample_df.iterrows():
 input_ids = torch.tensor(row['input_ids']).unsqueeze(0).to(model.device)
 att_mask = torch.tensor(row['attention_mask']).unsqueeze(0).to(model.device)
 logits = model(input_ids=input_ids, attention_mask=att_mask).logits.squeeze(0)
 sel_logits = logits[target_ids_tensor]
 preds.append(int(torch.argmax(sel_logits).item()))
 trues.append(labels_list[i])

 # 4) Metrics + full report (per-class + overall row)
 acc = accuracy_score(trues, preds)
 f1w = f1_score(trues, preds, average='weighted', zero_division=0)
 rep = classification_report(trues, preds, output_dict=True, zero_division=0)
 df = pd.DataFrame(rep).transpose().round(4).drop("accuracy", errors="ignore")
 acc_row = pd.DataFrame({'precision': [""],'recall': [""],'f1-score': [acc],'support': [sum(df["support"])]},
 index=["overall_accuracy"])
 final_df = pd.concat([df, acc_row])

 # 5) Save CSV (append with header tag)
 if report_path is None:
 report_path = os.path.join(GOEMOTIONS_PATH, f"per_class_silencing_class{class_id_local}.csv")
 os.makedirs(os.path.dirname(report_path), exist_ok=True)
 if experiment_title is None:
 experiment_title = f"Silencing top {percentage:.1%} neurons for class {class_id_local}"

 if not os.path.exists(report_path):
 with open(report_path, "w") as f:
 f.write(f"# {experiment_title}\n")
 final_df.to_csv(f)
 else:
 with open(report_path, "a") as f:
 f.write(f"\n\n# {experiment_title}\n")
 final_df.to_csv(report_path, mode="a")

 logger.info(f"Acc={acc:.4f} F1w={f1w:.4f} saved: {report_path}")

 # 6) Always cleanup
 for h in handles: h.remove()

 return {"accuracy": acc, "f1": f1w, "k_teor": k_teor, "k_real": k_real}

In [ ]:
percentages = SWEEP_PCTS # unified: single source of truth (control panel)

target_class_id = 4 # 0..5 (e.g., sadness)

for pct in percentages:
 silence_top_class_percentage_and_evaluate_goemo(
 model=model_goem,
 sample_df=sample_df,
 labels_list=labels_list,
 class_id_local=target_class_id,
 percentage=pct,
 experiment_title=f"Silencing {pct*100:.1f}% of neurons for class {target_class_id}",
 report_path=os.path.join(GOEMOTIONS_PATH, f"per_class_silencing_class{target_class_id}.csv")
 )

### FGSM

In [ ]:
from torch.nn import CrossEntropyLoss

def run_fgsm_attack_and_evaluate_goemo(
 model,
 sample_df,
 labels_list,
 epsilon: float = 0.1,
 report_path: str = None,
 experiment_title: str = None
):
 """
 FGSM on GoEmotions: compute grad wrt input embeddings, add sign(grad)*epsilon, re-run forward.
 Loss is computed on the 6-way slice (selected logits by target_ids_tensor) vs local labels (0..5).
 """
 if report_path is None:
 report_path = os.path.join(GOEMOTIONS_PATH, "fgsm.csv")
 if experiment_title is None:
 experiment_title = f"FGSM (ε={epsilon})"

 os.makedirs(os.path.dirname(report_path), exist_ok=True)
 loss_fn = CrossEntropyLoss()

 model.eval()
 predictions_fgsm = []

 logger.info(f"FGSM (ε={epsilon}) over {len(sample_df)} samples")
 for i in range(len(sample_df)):
 # inputs
 input_ids = torch.tensor(sample_df.loc[i, 'input_ids'], dtype=torch.long).unsqueeze(0).to(device_goemo)
 att_mask = torch.tensor(sample_df.loc[i, 'attention_mask'], dtype=torch.long).unsqueeze(0).to(device_goemo)
 y_local = torch.tensor([labels_list[i]], dtype=torch.long).to(device_goemo) # 0..5

 # get embeddings as leaf with grad
 with torch.no_grad():
 embeds0 = model.bert.embeddings(input_ids) # [1, seq, hidden]
 embeds = embeds0.detach().clone().requires_grad_(True)

 # forward on embeds (slice to 6-way)
 out = model(inputs_embeds=embeds, attention_mask=att_mask)
 logits = out.logits[:, target_ids_tensor] # [1, 6]
 loss = loss_fn(logits, y_local)

 # backward to get grad wrt embeds
 model.zero_grad(set_to_none=True)
 if embeds.grad is not None:
 embeds.grad.zero_()
 loss.backward()

 # FGSM perturbation
 adv_embeds = embeds + epsilon * embeds.grad.sign()

 # second forward with adversarial embeddings
 with torch.no_grad():
 out_adv = model(inputs_embeds=adv_embeds, attention_mask=att_mask)
 logits_adv = out_adv.logits[:, target_ids_tensor]
 pred = int(torch.argmax(logits_adv, dim=1).item())
 predictions_fgsm.append(pred)

 # cleanup
 del input_ids, att_mask, y_local, embeds0, embeds, adv_embeds, out, out_adv, logits, logits_adv, loss
 torch.cuda.empty_cache()

 # metrics + report (same shape you already use)
 accuracy = accuracy_score(labels_list, predictions_fgsm)
 f1w = f1_score(labels_list, predictions_fgsm, average='weighted', zero_division=0)

 report_dict = classification_report(labels_list, predictions_fgsm, output_dict=True, zero_division=0)
 report_df = pd.DataFrame(report_dict).transpose().round(4).drop("accuracy", errors="ignore")

 accuracy_row = pd.DataFrame({
 'precision': [""],
 'recall': [""],
 'f1-score': [accuracy],
 'support': [sum(report_df["support"])]
 }, index=["overall_accuracy"])
 final_df = pd.concat([report_df, accuracy_row])

 # save
 if not os.path.exists(report_path):
 with open(report_path, "w") as f:
 f.write(f"# {experiment_title}\n")
 final_df.to_csv(f)
 else:
 with open(report_path, "a") as f:
 f.write(f"\n\n# {experiment_title}\n")
 final_df.to_csv(report_path, mode="a")

 logger.info(f"Accuracy under FGSM (ε={epsilon}): {accuracy:.4f}")
 logger.info(f"Weighted F1 Score: {f1w:.4f}")
 logger.info(f"Classification report saved to {report_path}")

 return {"accuracy": accuracy, "f1": f1w, "report_df": final_df}

In [ ]:
for eps in [0.02, 0.05, 0.10, 0.15]:
 run_fgsm_attack_and_evaluate_goemo(
 model=model_goem,
 sample_df=sample_df,
 labels_list=labels_list,
 epsilon=eps,
 report_path=os.path.join(GOEMOTIONS_PATH, "fgsm.csv"),
 experiment_title=f"FGSM (ε={eps})"
 )


### Random Noise

In [ ]:
import os, logging, torch, pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report

logger = logging.getLogger(__name__)

def run_random_noise_attack_goemo(model, sample_df, labels_list, target_ids_tensor,
 epsilon=0.3, device=None,
 report_path=None, experiment_title=None, seed=None):
 """
 Add N(0, ε^2) noise to input embeddings and evaluate on the 6-class slice.
 Saves a full per-class report (CSV, append mode).
 """
 device = device or next(model.parameters()).device
 model.eval()
 preds, trues = [], []

 # optional reproducibility
 if seed is not None:
 import random, numpy as np
 random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
 torch.backends.cudnn.deterministic = True
 torch.backends.cudnn.benchmark = False

 is_longformer = hasattr(model, "longformer")

 with torch.no_grad():
 for i, row in sample_df.iterrows():
 input_ids = torch.tensor(row["input_ids"], dtype=torch.long, device=device).unsqueeze(0)
 att_mask = torch.tensor(row["attention_mask"], dtype=torch.long, device=device).unsqueeze(0)

 # embeddings + Gaussian noise
 embeds = model.base_model.embeddings(input_ids)
 if seed is not None:
 torch.manual_seed(seed + i)
 noise = torch.randn_like(embeds) * float(epsilon)
 noisy = embeds + noise

 if is_longformer:
 global_attention_mask = torch.zeros_like(att_mask)
 global_attention_mask[:, 0] = 1
 outputs = model(inputs_embeds=noisy, attention_mask=att_mask, global_attention_mask=global_attention_mask)
 else:
 outputs = model(inputs_embeds=noisy, attention_mask=att_mask)

 logits = outputs.logits.squeeze(0) # [28]
 sel_logits = logits[target_ids_tensor] # [6]
 pred_local = int(torch.argmax(sel_logits).item())
 preds.append(pred_local)
 trues.append(labels_list[i])

 # metrics + full report
 acc = accuracy_score(trues, preds)
 f1w = f1_score(trues, preds, average='weighted', zero_division=0)

 rep = classification_report(trues, preds, output_dict=True, zero_division=0)
 rep_df = pd.DataFrame(rep).transpose().round(4).drop("accuracy", errors="ignore")
 acc_row = pd.DataFrame({'precision': [""],'recall': [""],'f1-score': [acc],'support': [sum(rep_df["support"])]},
 index=["overall_accuracy"])
 final_df = pd.concat([rep_df, acc_row])

 # save CSV (append with header block)
 if report_path is None:
 report_path = os.path.join(GOEMOTIONS_PATH, "random_noise_report.csv")
 os.makedirs(os.path.dirname(report_path), exist_ok=True)
 if experiment_title is None:
 experiment_title = f"Random noise (ε={epsilon})"

 if not os.path.exists(report_path):
 with open(report_path, "w") as f:
 f.write(f"# {experiment_title}\n")
 final_df.to_csv(f)
 else:
 with open(report_path, "a") as f:
 f.write(f"\n\n# {experiment_title}\n")
 final_df.to_csv(report_path, mode="a")

 logger.info(f"Random noise ε={epsilon} Acc={acc:.4f} F1w={f1w:.4f} (saved: {report_path})")
 return {"accuracy": acc, "f1": f1w, "report_df": final_df, "preds": preds}


def sweep_random_noise_goemo(model, sample_df, labels_list, target_ids_tensor,
 epsilons, device=None, report_path=None, summary_path=None, seed=None):
 """
 Sweep over ε values, append per-class reports, and write a summary CSV with (epsilon, accuracy, f1_weighted).
 """
 device = device or next(model.parameters()).device
 if report_path is None:
 report_path = os.path.join(GOEMOTIONS_PATH, "random_noise_report.csv")
 if summary_path is None:
 summary_path = os.path.join(GOEMOTIONS_PATH, "random_noise_summary.csv")

 rows = []
 for eps in epsilons:
 res = run_random_noise_attack_goemo(
 model=model,
 sample_df=sample_df,
 labels_list=labels_list,
 target_ids_tensor=target_ids_tensor,
 epsilon=float(eps),
 device=device,
 report_path=report_path,
 experiment_title=f"Random noise (ε={eps:.3f})",
 seed=seed
 )
 rows.append({"epsilon": float(eps), "accuracy": res["accuracy"], "f1_weighted": res["f1"]})

 pd.DataFrame(rows).to_csv(summary_path, index=False)
 logger.info(f"Summary saved to {summary_path}")
 return rows

In [ ]:
# Barrida de epsilons
eps_grid = [0.02, 0.05, 0.10, 0.20, 0.40, 0.60, 0.80, 1.00, 1.20, 1.50]
report_csv = os.path.join(GOEMOTIONS_PATH, "random_noise_report.csv")
summary_csv = os.path.join(GOEMOTIONS_PATH, "random_noise_summary.csv")

_ = sweep_random_noise_goemo(
 model=model_goem,
 sample_df=sample_df,
 labels_list=labels_list, # 0..5 locales
 target_ids_tensor=target_ids_tensor,
 epsilons=eps_grid,
 device=device_goemo,
 report_path=report_csv,
 summary_path=summary_csv,
 seed=42
)

### Logit Bias

In [ ]:
import os, torch, pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report

def run_logit_bias_attack_goemo(
 model,
 sample_df,
 labels_list,
 target_ids_tensor, # Tensor con los 6 IDs globales del modelo original (28), en el orden del slice
 target_local_id: int, # 0..5 en el slice (p.ej., TARGET_EMOTIONS.index("joy"))
 delta: float = 1.0, # magnitud del sesgo (+) hacia la clase objetivo
 balanced_push: bool = False, # si True, reparte -delta/(K-1) en las otras 5 clases del slice
 report_path: str = None,
 experiment_title: str = None,
 device=None
):
 """
 Adds +delta to the target class logit (and optional -delta/(K-1) to the other slice logits) at inference time.
 Evaluation is done over the 6-class slice (target_ids_tensor).
 """
 if device is None:
 device = model.device if hasattr(model, "device") else torch.device("cpu")

 model.eval()
 preds = []
 K = int(target_ids_tensor.numel())
 assert 0 <= target_local_id < K, "target_local_id must be in [0..K-1]"

 # global id in the 28-way head
 global_target_id = int(target_ids_tensor[target_local_id].item())

 with torch.no_grad():
 for i, row in sample_df.iterrows():
 input_ids = torch.tensor(row["input_ids"]).unsqueeze(0).to(device)
 att_mask = torch.tensor(row["attention_mask"]).unsqueeze(0).to(device)

 logits = model(input_ids=input_ids, attention_mask=att_mask).logits # [1, 28]
 logits_mod = logits.clone()

 # +Δ to target
 logits_mod[0, global_target_id] += delta

 # Optional: push down others in the 6-way slice to keep things balanced
 if balanced_push and K > 1:
 other_mask = torch.ones(K, dtype=torch.bool, device=device)
 other_mask[target_local_id] = False
 other_global_ids = target_ids_tensor[other_mask] # length K-1
 logits_mod[0, other_global_ids] -= (delta / (K - 1))

 # Predict on the 6-class slice
 sel = logits_mod[0, target_ids_tensor] # [6]
 pred_local = int(torch.argmax(sel).item())
 preds.append(pred_local)

 # Metrics (slice 0..5)
 acc = accuracy_score(labels_list, preds)
 f1w = f1_score(labels_list, preds, average="weighted", zero_division=0)
 report_dict = classification_report(labels_list, preds, output_dict=True, zero_division=0)
 report_df = pd.DataFrame(report_dict).transpose().round(4)
 report_df = report_df.drop("accuracy", errors="ignore")

 accuracy_row = pd.DataFrame({
 'precision': [""],
 'recall': [""],
 'f1-score': [acc],
 'support': [sum(report_df["support"])]
 }, index=["overall_accuracy"])
 final_df = pd.concat([report_df, accuracy_row])

 # Save CSV
 if report_path is None:
 report_path = os.path.join(GOEMOTIONS_PATH, "logit_bias.csv")
 os.makedirs(os.path.dirname(report_path), exist_ok=True)

 if experiment_title is None:
 experiment_title = f"Logit Bias: class_local={target_local_id}, Δ={delta}, balanced={balanced_push}"

 if not os.path.exists(report_path):
 with open(report_path, "w") as f:
 f.write(f"# {experiment_title}\n")
 final_df.to_csv(f)
 else:
 with open(report_path, "a") as f:
 f.write(f"\n\n# {experiment_title}\n")
 final_df.to_csv(report_path, mode="a")

 logger.info(f"LogitBias class={target_local_id} Δ={delta} balanced={balanced_push} Acc={acc:.4f} F1w={f1w:.4f} (saved: {report_path})")
 return {"accuracy": acc, "f1": f1w, "report_df": final_df}

In [ ]:
import torch, numpy as np

def slice_margin_stats(model, sample_df, target_ids_tensor, target_local_id, device):
 # margin needed so that target beats the best other class in the 6-way slice
 model.eval()
 margins = []
 with torch.no_grad():
 for _, row in sample_df.iterrows():
 input_ids = torch.tensor(row["input_ids"]).unsqueeze(0).to(device)
 att_mask = torch.tensor(row["attention_mask"]).unsqueeze(0).to(device)
 logits = model(input_ids=input_ids, attention_mask=att_mask).logits[0]
 sel = logits[target_ids_tensor] # shape [6]
 target_logit = sel[target_local_id].item()
 best_other = torch.max(torch.cat([sel[:target_local_id], sel[target_local_id+1:]])).item()
 margins.append(best_other - target_logit) # if >0, target needs at least this Δ to win
 margins = np.array(margins)
 q = np.quantile(margins, [0.5, 0.8, 0.9, 0.95])
 print(f"Δ needed (median, p80, p90, p95): {q}")
 return margins

# Example: calibrate for "joy"
target_local_id = TARGET_EMOTIONS.index("joy")
_ = slice_margin_stats(model_goem, sample_df, target_ids_tensor, target_local_id, device_goemo)

In [ ]:
# target_local_id = TARGET_EMOTIONS.index("joy") # ajusta la emoción objetivo
for delta in [8, 9, 10, 11, 12]:
 run_logit_bias_attack_goemo(
 model=model_goem,
 sample_df=sample_df,
 labels_list=labels_list,
 target_ids_tensor=target_ids_tensor,
 target_local_id=target_local_id,
 delta=delta,
 balanced_push=False, # clave
 report_path=os.path.join(GOEMOTIONS_PATH, "logit_bias.csv"),
 experiment_title=f"LogitBias Δ={delta} (balanced=False)",
 device=device_goemo
 )

### Weight Space Attack

In [ ]:
import os, torch, numpy as np, pandas as pd
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score, classification_report

# --- helpers ---
def _topk_for_goemo(probe, percentage, label2idx, target_local_class, class_specific=True):
 # returns global neuron indices (0..H*L-1)
 if class_specific:
 _, per_class = linear_probe.get_top_neurons(probe, percentage=percentage, class_to_idx=label2idx)
 arr = per_class[target_local_class]
 else:
 topk_global, _ = linear_probe.get_top_neurons(probe, percentage=percentage, class_to_idx=label2idx)
 arr = topk_global
 return [int(x) for x in np.asarray(arr).ravel().tolist()]

def _cols_from_global_indices(global_idx, hidden, num_layers, layers_scope):
 if layers_scope not in ("all", "last_only"):
 raise ValueError("layers_scope must be 'all' or 'last_only'")
 if layers_scope == "last_only":
 last = num_layers - 1
 global_idx = [g for g in global_idx if last*hidden <= g < (last+1)*hidden]
 # map to hidden dims (columns in the head)
 return sorted({ g % hidden for g in global_idx })


# --- main attack with diagnostics (logs como en malware) ---
def weight_head_push_goemo_diag(
 target_local_class, # 0..5 in your 6-way slice
 percentage=0.10,
 delta_scale=0.2,
 class_specific=True,
 layers_scope="all", # "all"or "last_only"
 max_cols=200, # None = no cap
 bias_only=False,
 balanced_push=False,
 balanced_push_factor=1.0,
 suppress_local_class=None, # another local class 0..5 to push down
 suppress_factor=0.5,
 report_path=None,
 experiment_title=None
):
 model = model_goem
 hidden = model.config.hidden_size
 num_layers = model.config.num_hidden_layers

 # local(0..5) global(28) row in the head
 target_global = int(target_ids_tensor[target_local_class].item())
 suppress_global = None
 if suppress_local_class is not None:
 suppress_global = int(target_ids_tensor[suppress_local_class].item())

 # 1) select columns from probe-driven indices
 topk_global = _topk_for_goemo(probe, percentage, label2idx, target_local_class, class_specific=class_specific)
 cols_all = _cols_from_global_indices(topk_global, hidden, num_layers, layers_scope)
 cols = cols_all[:max_cols] if (max_cols is not None) else cols_all
 if len(cols) == 0:
 logger.warning("[WeightPush-GoEmo] No columns selected.")
 return None

 # 2) classifier head (BERT SequenceClassification: model.classifier)
 clf = get_classifier_linear(model) # ya la tienes definida en tu código
 W, b = clf.weight, clf.bias # [28, H]
 C, H = W.shape

 logger.info(
 f"[WeightPush-GoEmo] local={target_local_class}global={target_global} "
 f"p={percentage:.0%} cols={len(cols)} Δ={delta_scale} "
 f"bias_only={bias_only} balanced={balanced_push} "
 f"suppress_local={suppress_local_class} ({suppress_global})"
 )

 # 3) backup
 W_orig = W.data.clone()
 b_orig = b.data.clone() if b is not None else None

 try:
 # 4) temporary edit
 if bias_only:
 if b is None: raise RuntimeError("Classifier has no bias.")
 delta_b = torch.zeros_like(b.data)
 delta_b[target_global] += delta_scale
 if balanced_push and C > 1:
 neg = (delta_scale * balanced_push_factor) / (C - 1)
 mask = torch.ones(C, dtype=torch.bool, device=b.device)
 mask[target_global] = False
 delta_b[mask] -= neg
 if suppress_global is not None and suppress_global != target_global:
 delta_b[suppress_global] -= (delta_scale * suppress_factor)
 b.data.add_(delta_b.to(b.device))
 else:
 delta = torch.zeros_like(W.data) # [C, H]
 delta[target_global, cols] += delta_scale
 if balanced_push and C > 1:
 neg = (delta_scale * balanced_push_factor) / (C - 1)
 mask = torch.ones(C, dtype=torch.bool, device=W.device)
 mask[target_global] = False
 delta[mask][:, cols] -= neg
 if suppress_global is not None and suppress_global != target_global:
 delta[suppress_global, cols] -= (delta_scale * suppress_factor)
 W.data.add_(delta.to(W.device))

 # 5) inference on 6-way slice
 model.eval()
 preds_local, trues_local = [], []
 with torch.no_grad():
 for i, row in sample_df.iterrows():
 input_ids = torch.tensor(row["input_ids"]).unsqueeze(0).to(device_goemo)
 att_mask = torch.tensor(row["attention_mask"]).unsqueeze(0).to(device_goemo)
 logits28 = model(input_ids=input_ids, attention_mask=att_mask).logits.squeeze(0)
 sel = logits28[target_ids_tensor] # 6 logits (your subset)
 pred_local = int(torch.argmax(sel).item()) # 0..5
 preds_local.append(pred_local)
 trues_local.append(int(labels_list[i]))

 finally:
 # 6) restore
 W.data.copy_(W_orig)
 if b is not None and b_orig is not None:
 b.data.copy_(b_orig)

 # 7) diagnostics (mismo estilo que malware)
 acc = accuracy_score(trues_local, preds_local)
 f1w = f1_score(trues_local, preds_local, average='weighted', zero_division=0)
 logger.info(f"[WeightPush-GoEmo] Accuracy: {acc:.4f}")
 logger.info(f"[WeightPush-GoEmo] Weighted F1: {f1w:.4f}")

 dist = dict(Counter(preds_local))
 logger.info(f"[WeightPush-GoEmo] Prediction distribution (local ids): {dist}")

 mapping = Counter(zip(trues_local, preds_local))
 mapping_full = {f"{o}{p}": c for (o, p), c in mapping.items()}
 logger.info(f"[WeightPush-GoEmo] Mapping originalattacked (local ids): {mapping_full}")

 to_target = {k: v for k, v in mapping_full.items() if k.endswith(f"{target_local_class}")}
 flips_to_target = sum(
 c for k, c in mapping_full.items()
 if k.split("")[0] != str(target_local_class) and k.endswith(f"{target_local_class}")
 )
 kept_as_target = mapping.get((target_local_class, target_local_class), 0)

 total_non_target = sum(1 for y in trues_local if y != target_local_class)
 frac_flips_from_non_target = (flips_to_target / total_non_target) if total_non_target else 0.0
 frac_all_to_target = (sum(to_target.values()) / len(trues_local)) if trues_local else 0.0

 logger.info(f"[WeightPush-GoEmo] ONLY to target {target_local_class}: {to_target}")
 logger.info(f"[WeightPush-GoEmo] Flipstarget (from other classes): {flips_to_target}")
 logger.info(f"[WeightPush-GoEmo] Kept as target (targettarget): {kept_as_target}")
 logger.info(f"[WeightPush-GoEmo] Frac non-target flippedtarget: {frac_flips_from_non_target:.2%}")
 logger.info(f"[WeightPush-GoEmo] Overall frac predicted as target: {frac_all_to_target:.2%}")

 # 8) optional CSV (formato igual que usas)
 if report_path:
 os.makedirs(os.path.dirname(report_path), exist_ok=True)
 if experiment_title is None:
 experiment_title = (f"WeightPush local={target_local_class} p={percentage:.0%} "
 f"Δ={delta_scale} cols={len(cols)} balanced={balanced_push} "
 f"bias_only={bias_only}")
 rep = classification_report(trues_local, preds_local, output_dict=True, zero_division=0)
 df = pd.DataFrame(rep).transpose().round(4).drop("accuracy", errors="ignore")
 acc_row = pd.DataFrame({'precision': [""],'recall': [""],'f1-score': [acc],'support':[sum(df["support"])]},
 index=["overall_accuracy"])
 final_df = pd.concat([df, acc_row])
 if not os.path.exists(report_path):
 with open(report_path, "w") as f:
 f.write(f"# {experiment_title}\n")
 final_df.to_csv(f)
 else:
 with open(report_path, "a") as f:
 f.write(f"\n\n# {experiment_title}\n")
 final_df.to_csv(report_path, mode="a")
 logger.info(f"[WeightPush-GoEmo] Report saved to {report_path}")

 return {
 "accuracy": acc,
 "f1_weighted": f1w,
 "prediction_distribution": dist,
 "mapping_full": mapping_full,
 "only_to_target": to_target,
 "flips_to_target": flips_to_target,
 "kept_as_target": kept_as_target,
 "frac_flips_from_non_target": frac_flips_from_non_target,
 "frac_all_to_target": frac_all_to_target,
 "used_columns": cols,
 "params": dict(
 target_local_class=target_local_class, percentage=percentage, delta_scale=delta_scale,
 class_specific=class_specific, layers_scope=layers_scope, max_cols=max_cols,
 bias_only=bias_only, balanced_push=balanced_push, balanced_push_factor=balanced_push_factor,
 suppress_local_class=suppress_local_class, suppress_factor=suppress_factor
 ),
 }

In [ ]:
def get_classifier_linear(model):
 """
 Return the final linear classifier layer (weight,bias) for HF sequence classifiers.
 Works with BERT/GoEmotions (model.classifier) and common fallbacks.
 """
 # Standard HF heads (e.g., BertForSequenceClassification)
 if hasattr(model, "classifier"):
 head = model.classifier
 if hasattr(head, "out_proj"): # e.g., some heads wrap a Linear as out_proj
 return head.out_proj
 if hasattr(head, "weight") and hasattr(head, "bias"):
 return head

 # Some architectures expose .score
 if hasattr(model, "score"):
 return model.score

 raise NotImplementedError("Could not find a linear classification head with (weight,bias).")

In [ ]:
# === Weight-head push: example runs (GoEmotions, local ids 0..5) ===
import os

TARGET_LOCAL = 3 # e.g., 3 = "joy"in your local mapping

# 1) Balanced push: target up, others slightly down (factor=0.5)
res_balanced = weight_head_push_goemo_diag(
 target_local_class=TARGET_LOCAL,
 percentage=0.20, # top-20% hidden dims (by probe) collapsed to unique columns
 delta_scale=0.2, # magnitude of the weight edit
 class_specific=True, # use per-class probe ranking
 layers_scope="all", # select from all layers (columns = hidden dims)
 max_cols=200, # cap number of edited columns
 bias_only=False, # edit weights (not only bias)
 balanced_push=True, # decrease others slightly
 balanced_push_factor=0.5, # strength of the decrease on non-target rows
 report_path=os.path.join(GOEMOTIONS_PATH, "weight_attack.csv"),
 experiment_title=f"WeightPush balanced: class={TARGET_LOCAL} p=20% cols=200 Δ=0.2 factor=0.5"
)





In [ ]:
# 2) Unbalanced push + explicit suppression of a competing class (e.g., sadness=4)
res_suppress = weight_head_push_goemo_diag(
 target_local_class=TARGET_LOCAL,
 percentage=0.25,
 delta_scale=0.5,
 class_specific=True,
 layers_scope="all",
 max_cols=250,
 bias_only=False,
 balanced_push=False, # no generic down-push to others
 suppress_local_class=4, # explicitly suppress a rival class
 suppress_factor=0.7, # suppression strength
 report_path=os.path.join(GOEMOTIONS_PATH, "weight_attack.csv"),
 experiment_title=f"WeightPush suppress: class={TARGET_LOCAL} p=25% cols=250 Δ=0.5 suppress=4×0.7"
)

In [ ]:
# 3) Bias-only variant (no weight edits; percentage ignored)
res_bias_only = weight_head_push_goemo_diag(
 target_local_class=TARGET_LOCAL,
 percentage=0.0, # ignored when bias_only=True
 delta_scale=0.6, # bias delta
 class_specific=True,
 layers_scope="all",
 max_cols=None,
 bias_only=True, # edit bias only
 balanced_push=True,
 balanced_push_factor=0.5,
 report_path=os.path.join(GOEMOTIONS_PATH, "weight_attack.csv"),
 experiment_title=f"WeightPush bias-only: class={TARGET_LOCAL} Δ_bias=0.6 factor=0.5"
)

# Compact summary print
def _summ(res):
 if not res:
 return "—"
 return (
 f"Acc={res['accuracy']:.4f} | F1w={res['f1_weighted']:.4f} | "
 f"flipstarget={res['flips_to_target']} | kept_target={res['kept_as_target']}"
 )

print("Balanced push ", _summ(res_balanced))
print("Suppress rival ", _summ(res_suppress))
print("Bias-only ", _summ(res_bias_only))

### Global Activation Noise (Gaussian)

In [ ]:
# --- Global Activation Noise (Gaussian) over CLS, across all encoder layers ---
import os, logging, torch, pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report

logger = logging.getLogger(__name__)

def make_cls_additive_noise_hook(sigma: float):
 """Add N(0, sigma^2) noise to the CLS vector."""
 def hook(_module, _inp, out):
 if not isinstance(out, torch.Tensor):
 return out
 out2 = out.clone()
 if out2.dim() == 3:
 # (batch, seq_len, hidden)
 cls = out2[:, 0, :]
 cls = cls + torch.randn_like(cls) * sigma
 out2[:, 0, :] = cls
 elif out2.dim() == 2:
 # (batch, hidden)
 out2 = out2 + torch.randn_like(out2) * sigma
 return out2
 return hook

def run_global_activation_noise_goemo(
 sigma: float = 0.1,
 report_path: str = None,
 experiment_title: str = None,
):
 model = model_goem
 model.eval()

 # 1) Register hooks on all encoder layers (output.LayerNorm) + pooler (optional)
 handles = []
 for L in range(model.config.num_hidden_layers):
 h = model.bert.encoder.layer[L].output.LayerNorm.register_forward_hook(
 make_cls_additive_noise_hook(sigma)
 )
 handles.append(h)
 # (optional) pooler: uncomment if you also want noise there
 # handles.append(model.bert.pooler.dense.register_forward_hook(make_cls_additive_noise_hook(sigma)))

 # 2) Inference on the 6-way slice
 preds, trues = [], []
 with torch.no_grad():
 for i, row in sample_df.iterrows():
 input_ids = torch.tensor(row["input_ids"]).unsqueeze(0).to(device_goemo)
 att_mask = torch.tensor(row["attention_mask"]).unsqueeze(0).to(device_goemo)
 logits = model(input_ids=input_ids, attention_mask=att_mask).logits.squeeze(0)
 sel_logits = logits[target_ids_tensor]
 pred_local = int(torch.argmax(sel_logits).item())
 preds.append(pred_local)
 trues.append(labels_list[i])

 # 3) Metrics + full report
 acc = accuracy_score(trues, preds)
 f1w = f1_score(trues, preds, average="weighted", zero_division=0)
 report_dict = classification_report(trues, preds, output_dict=True, zero_division=0)
 df = pd.DataFrame(report_dict).transpose().round(4).drop("accuracy", errors="ignore")
 acc_row = pd.DataFrame({
 "precision": [""], "recall": [""], "f1-score": [acc], "support": [df["support"].sum()]
 }, index=["overall_accuracy"])
 final_df = pd.concat([df, acc_row])

 # 4) Save CSV (append with a header)
 if report_path is None:
 report_path = os.path.join(GOEMOTIONS_PATH, "global_activation_noise.csv")
 os.makedirs(os.path.dirname(report_path), exist_ok=True)
 if experiment_title is None:
 experiment_title = f"Global activation noise (sigma={sigma})"

 if not os.path.exists(report_path):
 with open(report_path, "w") as f:
 f.write(f"# {experiment_title}\n")
 final_df.to_csv(f)
 else:
 with open(report_path, "a") as f:
 f.write(f"\n\n# {experiment_title}\n")
 final_df.to_csv(report_path, mode="a")

 logger.info(f"Global activation noise σ={sigma} Acc={acc:.4f} F1w={f1w:.4f} (saved: {report_path})")

 # 5) Remove hooks
 for h in handles: h.remove()

 return {"accuracy": acc, "f1": f1w, "report_df": final_df}

In [ ]:
sigmas = [0.02, 0.05, 0.1, 0.2, 0.4]
for s in sigmas:
 run_global_activation_noise_goemo(
 sigma=s,
 report_path=os.path.join(GOEMOTIONS_PATH, "global_activation_noise.csv"),
 experiment_title=f"Global activation noise (sigma={s})"
 )

### Fault Sneaking (sim)

In [ ]:
# --- Fault Sneaking: attenuation of selected CLS dims (global or per-class) ---
import numpy as np
from collections import defaultdict

def _top_neurons_global_goemo(percentage: float):
 top_global, _ = linear_probe.get_top_neurons(probe, percentage=percentage, class_to_idx=label2idx)
 return [int(x) for x in np.asarray(top_global).ravel().tolist()]

def _top_neurons_for_class_goemo(class_id: int, percentage: float):
 _, per_class = linear_probe.get_top_neurons(probe, percentage=percentage, class_to_idx=label2idx)
 arr = per_class[class_id]
 return [int(x) for x in np.asarray(arr).ravel().tolist()]

def _map_global_to_layer(local_model, global_indices, layers_scope="all"):
 hidden = local_model.config.hidden_size
 L = local_model.config.num_hidden_layers
 if layers_scope not in ("all", "last_only"):
 raise ValueError("layers_scope must be 'all' or 'last_only'")
 if layers_scope == "last_only":
 last = L - 1
 global_indices = [g for g in global_indices if last*hidden <= g < (last+1)*hidden]

 layer_to_idx = defaultdict(list)
 for g in global_indices:
 layer = g // hidden
 dim = g % hidden
 if 0 <= layer < L:
 layer_to_idx[layer].append(dim)
 return layer_to_idx

def make_cls_attenuation_hook(indices, alpha: float):
 """Multiply selected CLS dims by (1 - alpha)."""
 scale = 1.0 - float(alpha)
 idx_tensor = torch.tensor(indices, dtype=torch.long) if indices else torch.tensor([], dtype=torch.long)
 def hook(_m, _i, out):
 if not isinstance(out, torch.Tensor): return out
 o = out.clone()
 if o.dim() == 3:
 cls = o[:, 0, :]
 if idx_tensor.numel() == 0:
 cls = cls * scale
 else:
 idx = idx_tensor.to(o.device)
 cls[:, idx] = cls[:, idx] * scale
 o[:, 0, :] = cls
 elif o.dim() == 2:
 if idx_tensor.numel() == 0:
 o = o * scale
 else:
 idx = idx_tensor.to(o.device)
 o[:, idx] = o[:, idx] * scale
 return o
 return hook

def run_fault_sneaking_goemo(
 percentage: float = 0.10, # fraction of dims to attenuate (if using top-k)
 alpha: float = 0.2, # attenuation strength (0.2 keep 80%)
 selection: str = "global", # "global"or "per_class"
 class_id: int = None, # required if selection="per_class"(0..5 local id)
 layers_scope: str = "all",
 report_path: str = None,
 experiment_title: str = None
):
 model = model_goem
 model.eval()

 # 1) Select neurons (global or per-class)
 if selection == "global":
 global_idx = _top_neurons_global_goemo(percentage)
 elif selection == "per_class":
 if class_id is None:
 raise ValueError("class_id is required when selection='per_class'")
 global_idx = _top_neurons_for_class_goemo(class_id, percentage)
 else:
 raise ValueError("selection must be 'global' or 'per_class'")

 layer_map = _map_global_to_layer(model, global_idx, layers_scope=layers_scope)

 # 2) Register hooks
 handles = []
 for L, dims in layer_map.items():
 h = model.bert.encoder.layer[L].output.LayerNorm.register_forward_hook(
 make_cls_attenuation_hook(dims, alpha)
 )
 handles.append(h)

 # 3) Inference
 preds, trues = [], []
 with torch.no_grad():
 for i, row in sample_df.iterrows():
 input_ids = torch.tensor(row["input_ids"]).unsqueeze(0).to(device_goemo)
 att_mask = torch.tensor(row["attention_mask"]).unsqueeze(0).to(device_goemo)
 logits = model(input_ids=input_ids, attention_mask=att_mask).logits.squeeze(0)
 sel_logits = logits[target_ids_tensor]
 pred_local = int(torch.argmax(sel_logits).item())
 preds.append(pred_local)
 trues.append(labels_list[i])

 # 4) Metrics + report
 acc = accuracy_score(trues, preds)
 f1w = f1_score(trues, preds, average="weighted", zero_division=0)
 rep = classification_report(trues, preds, output_dict=True, zero_division=0)
 df = pd.DataFrame(rep).transpose().round(4).drop("accuracy", errors="ignore")
 acc_row = pd.DataFrame({
 "precision": [""], "recall": [""], "f1-score": [acc], "support": [df["support"].sum()]
 }, index=["overall_accuracy"])
 final_df = pd.concat([df, acc_row])

 # 5) Save CSV
 if report_path is None:
 report_path = os.path.join(GOEMOTIONS_PATH, "fault_sneaking.csv")
 os.makedirs(os.path.dirname(report_path), exist_ok=True)
 if experiment_title is None:
 tag = f"{selection}"+ (f"_c{class_id}"if selection=="per_class"else "")
 experiment_title = f"Fault sneaking ({tag}) p={percentage:.0%} alpha={alpha} scope={layers_scope}"

 if not os.path.exists(report_path):
 with open(report_path, "w") as f:
 f.write(f"# {experiment_title}\n")
 final_df.to_csv(f)
 else:
 with open(report_path, "a") as f:
 f.write(f"\n\n# {experiment_title}\n")
 final_df.to_csv(report_path, mode="a")

 logger.info(f"Fault sneaking [{selection}] p={percentage:.0%} α={alpha} Acc={acc:.4f} F1w={f1w:.4f} (saved: {report_path})")

 # 6) Remove hooks
 for h in handles: h.remove()

 return {"accuracy": acc, "f1": f1w, "report_df": final_df}

In [ ]:
# (A) Global attenuation over top-k dims (probe-guided)
for a in [0.1, 0.2, 0.4]:
 run_fault_sneaking_goemo(
 percentage=0.20, # 20% of dims
 alpha=a, # attenuation strength
 selection="global",
 layers_scope="all",
 report_path=os.path.join(GOEMOTIONS_PATH, "fault_sneaking.csv"),
 experiment_title=f"Fault sneaking (global) p=20% alpha={a}"
 )

# (B) Per-class attenuation (e.g., target local class=3)
for a in [0.1, 0.2, 0.4]:
 run_fault_sneaking_goemo(
 percentage=0.20,
 alpha=a,
 selection="per_class",
 class_id=3,
 layers_scope="all",
 report_path=os.path.join(GOEMOTIONS_PATH, "fault_sneaking.csv"),
 experiment_title=f"Fault sneaking (per_class=3) p=20% alpha={a}"
 )

In [ ]:
# === Save profiling (GoEmotions) ===
_gp_dir = os.path.dirname(ACTIVATIONS_GOEMOTIONS)
os.makedirs(f"{_gp_dir}/results", exist_ok=True)
print(f"[profile] GoEmotions TOTAL wall-clock: {time.perf_counter()-PROFILE_T0:.1f}s")
PROFILE.save(f"{_gp_dir}/results/complexity_time_memory_goemotions.csv")
